# Risk Analyzer: upload meeting minutes, chat with them, review the evidence-backed risk register, export a report
**Works in Google Colab and Jupyter. Convert to a `.py` engine for Streamlit** (see the last cell). Every cell below the line
"NOTEBOOK DEMO" is optional demo UI; everything above it is the reusable engine (functions only, no side effects).

Pipeline (LangGraph, 10 nodes + human review): intake -> PDF extraction/OCR -> metadata parsing -> chunk + index (ChromaDB) ->
evidence retrieval (RAG) -> risk & action extraction (LLM) -> classification -> historical matching -> quality check ->
**human review (graph pauses)** -> report/export.

*AI-generated assessments are decision support, not a substitute for professional judgement.*

## 1. Setup (installs what is missing; works on Colab and local Jupyter)

In [ ]:
import os, sys, re, json, glob, time, shutil, hashlib, difflib, operator, copy, subprocess, warnings, importlib.util
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
try:
    get_ipython(); IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False

INSTALL_DEPS = True      # set False once everything is installed (e.g. in your Streamlit environment)
REQUIRED_PIP = {"langgraph": "langgraph", "pydantic": "pydantic", "pypdf": "pypdf", "pdfplumber": "pdfplumber", "pypdfium2": "pypdfium2",
                "pytesseract": "pytesseract", "sklearn": "scikit-learn", "pandas": "pandas", "matplotlib": "matplotlib", "reportlab": "reportlab",
                "numpy": "numpy", "ipywidgets": "ipywidgets", "PIL": "pillow<13", "langchain_core": "langchain-core",
                "langchain_community": "langchain-community", "langchain_text_splitters": "langchain-text-splitters",
                "langchain_chroma": "langchain-chroma", "chromadb": "chromadb", "langchain_openai": "langchain-openai",
                "langchain_anthropic": "langchain-anthropic"}

def ensure_deps():
    missing = [pip for mod, pip in REQUIRED_PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        # If Chroma complains about opentelemetry versions, pin them:  pip install -U "opentelemetry-api==1.44.0" "opentelemetry-sdk==1.44.0"
    if shutil.which("tesseract") is None:
        if IN_COLAB:
            subprocess.run(["apt-get", "-q", "install", "-y", "tesseract-ocr"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        else:
            print("NOTE: Tesseract OCR not found, so scanned PDFs cannot be read. Windows: https://github.com/UB-Mannheim/tesseract/wiki | "
                  "macOS: brew install tesseract | Linux: apt-get install tesseract-ocr")
if INSTALL_DEPS: ensure_deps()

from pathlib import Path
from datetime import datetime
from typing import TypedDict, List, Dict, Optional, Annotated, Any
import numpy as np
import pandas as pd
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

## 2. Configuration, API keys, audit log

In [ ]:
WORK = "/content/risk_workspace" if IN_COLAB else "risk_workspace"
CFG = dict(
    WORK_DIR=WORK, UPLOAD_DIR=f"{WORK}/uploads", OUT_DIR=f"{WORK}/outputs", CHROMA_DIR=f"{WORK}/chroma", CACHE_DIR=f"{WORK}/cache",
    LLM_PROVIDER="auto",            # "auto" | "openai" | "anthropic" | "mock"   (auto = whichever API key is set, else mock)
    OPENAI_MODEL="gpt-4o-mini", ANTHROPIC_MODEL="claude-sonnet-5-5",
    EMBEDDER="auto",                # "auto" | "openai" (text-embedding-3-small) | "local" (offline hashing embeddings)
    EMBED_MODEL="text-embedding-3-small",
    PDF_LOADER="pypdf",             # "pypdf" (PyPDFLoader, layout mode) | "pdfplumber"
    CHUNK_SIZE=1000, CHUNK_OVERLAP=200, TOP_K=4,
    MATCH_THRESHOLD=None,           # None = automatic for the chosen embedder
    CONF_MIN=0.6, MAX_RETRIES=1, MIN_TEXT_CHARS=50, MAX_FILE_MB=50,
    HUMAN_REVIEW=True,              # graph pauses for the project manager before the final report
)
LOCAL_MATCH_THRESHOLD, OPENAI_MATCH_THRESHOLD = 0.06, 0.55     # similarity needed to link a new mention to an existing register risk (tune on your data)
for _k in ("UPLOAD_DIR", "OUT_DIR", "CHROMA_DIR", "CACHE_DIR"): Path(CFG[_k]).mkdir(parents=True, exist_ok=True)
DISCLAIMER = "AI-generated assessments are decision support, not a substitute for professional judgement."

CATEGORIES = ["Cost", "Schedule", "Supply Chain", "Technical", "Resource", "Regulatory", "Contractual", "Quality", "Safety", "Environmental", "Stakeholder"]
STAGES = ["raised", "escalated", "mitigating", "closed", "open"]
TYPES = ["Risk", "Issue"]
STRATEGIES = ["Avoid", "Transfer", "Mitigate", "Accept", "Escalate", "Not stated"]      # PMI threat responses
RUBRIC = """Likelihood (1-5): 1 rare; 2 unlikely; 3 possible; 4 likely / already showing signs; 5 almost certain or occurring now.
Impact (1-5). Rate the effect on schedule and on cost separately and use the HIGHER rating.
Schedule (delay to substantial completion): 1 <= 2 working days; 2 = 3-5 days; 3 = 6-10 days; 4 = 11-20 days; 5 = more than 20 days or a contract milestone missed.
Cost (% of original contract value): 1 < 0.25%; 2 = 0.25-0.5%; 3 = 0.5-1%; 4 = 1-2%; 5 = more than 2%.
If the minutes give neither a delay nor a cost figure, estimate the likely effect and say so in the rationale. Never invent figures.
Score = Probability x Impact (PMI). Likelihood 1-5 means probability 0.1 / 0.3 / 0.5 / 0.7 / 0.9, so the maximum score is 4.5. Red >= 2.5, Amber 1.2-2.4, Green <= 1.0. Closed risks get Likelihood 1."""
PROB = {1: 0.1, 2: 0.3, 3: 0.5, 4: 0.7, 5: 0.9}            # PMI probability scale. The model still proposes likelihood 1-5; the code converts it
def score_of(likelihood, impact): return round(PROB[int(likelihood)] * int(impact), 1)        # 0.1 to 4.5
def rag_of(score): return "Red" if score >= 2.5 else "Amber" if score >= 1.1 else "Green"

def load_keys():
    for name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        if not os.environ.get(name) and IN_COLAB:
            try:
                from google.colab import userdata
                v = userdata.get(name)
                if v: os.environ[name] = v
            except Exception: pass

_VS = None; _LLM = None; _EMB = None
def resolve_providers(verbose=True):
    global _VS, _LLM, _EMB
    load_keys(); _VS = _LLM = _EMB = None
    p, e = CFG["LLM_PROVIDER"], CFG["EMBEDDER"]
    if p == "auto": p = "openai" if os.environ.get("OPENAI_API_KEY") else "anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "mock"
    if e == "auto": e = "openai" if os.environ.get("OPENAI_API_KEY") else "local"
    CFG["_llm"], CFG["_embedder"] = p, e
    CFG["_thr"] = CFG["MATCH_THRESHOLD"] or (OPENAI_MATCH_THRESHOLD if e == "openai" else LOCAL_MATCH_THRESHOLD)
    if verbose: print(f"LLM provider: {p} | embeddings: {e} | PDF loader: {CFG['PDF_LOADER']}" + ("   (no API key found: running in evidence-only 'mock' mode)" if p == "mock" else ""))

def set_api_key(name, value):
    """e.g. set_api_key("OPENAI_API_KEY", "sk-...")  then the providers are re-detected."""
    os.environ[name] = value; resolve_providers()

# ---- audit log: extraction failures, unsupported claims, inconsistencies, user corrections
AUDIT: List[dict] = []
def log_event(kind, doc_id=None, detail="", **extra):
    ev = dict(ts=datetime.now().isoformat(timespec="seconds"), kind=kind, doc_id=doc_id, detail=detail, **extra)
    AUDIT.append(ev)
    try:
        with open(Path(CFG["OUT_DIR"]) / "audit_log.jsonl", "a", encoding="utf-8") as f: f.write(json.dumps(ev, default=str) + "\n")
    except Exception: pass
    return ev
resolve_providers()

## 3. LLM + embeddings + ChromaDB vector store

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document

class LocalHashEmbeddings(Embeddings):
    """Offline embeddings (no API key). Fixed size, so it works with ChromaDB."""
    def __init__(self, n=2048):
        from sklearn.feature_extraction.text import HashingVectorizer
        self.v = HashingVectorizer(n_features=n, ngram_range=(1, 2), alternate_sign=False, norm="l2", stop_words="english")
    def embed_documents(self, texts): return self.v.transform(texts).toarray().tolist()
    def embed_query(self, text): return self.embed_documents([text])[0]

def get_embeddings():
    global _EMB
    if _EMB is None:
        if CFG["_embedder"] == "openai":
            from langchain_openai import OpenAIEmbeddings
            _EMB = OpenAIEmbeddings(model=CFG["EMBED_MODEL"])
        else: _EMB = LocalHashEmbeddings()
    return _EMB

def get_vectorstore():
    global _VS
    if _VS is None:
        from langchain_chroma import Chroma
        _VS = Chroma(collection_name=f"minutes_{CFG['_embedder']}", embedding_function=get_embeddings(), persist_directory=CFG["CHROMA_DIR"])
    return _VS

def get_llm():
    global _LLM
    if _LLM is None and CFG["_llm"] != "mock":
        if CFG["_llm"] == "openai":
            from langchain_openai import ChatOpenAI
            _LLM = ChatOpenAI(model=CFG["OPENAI_MODEL"], temperature=0)
        else:
            from langchain_anthropic import ChatAnthropic
            _LLM = ChatAnthropic(model=CFG["ANTHROPIC_MODEL"], temperature=0, max_tokens=4096)
    return _LLM

def llm_text(system, user):
    llm = get_llm()
    if llm is None: return None
    try:
        r = llm.invoke([("system", system), ("human", user)]).content
        return r if isinstance(r, str) else "".join(b.get("text", "") for b in r if isinstance(b, dict))
    except Exception as e:
        log_event("llm_failure", detail=f"{type(e).__name__}: {e}"); print("LLM error:", e); return None

def _parse_json(txt):
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    return json.loads(txt[txt.find("{"): txt.rfind("}") + 1])

def llm_json(system, user, doc_id=None):
    """Parsed JSON dict, or None (mock mode / repeated failure) so callers use their rule-based fallback."""
    if CFG["_llm"] == "mock": return None
    for attempt in range(2):
        txt = llm_text(system, user)
        if txt is None: return None
        try: return _parse_json(txt)
        except Exception as e: log_event("llm_failure", doc_id, f"invalid JSON (attempt {attempt+1}): {e}")
    return None

_VEC: Dict[str, np.ndarray] = {}
def embed_matrix(texts):
    need = [t for t in dict.fromkeys(texts) if t not in _VEC]
    if need:
        for t, v in zip(need, get_embeddings().embed_documents(need)):
            a = np.asarray(v, dtype=float); n = np.linalg.norm(a); _VEC[t] = a / n if n else a
    return np.stack([_VEC[t] for t in texts])

## 4. PDF loading (PyPDFLoader + OCR fallback), parsing helpers, numeric checks

In [ ]:
def load_pdf_pages(path):
    """-> [{'page': 1-based, 'text': str, 'method': 'text'|'ocr'}]. Scanned pages go through OCR."""
    if CFG["PDF_LOADER"] == "pypdf":
        from langchain_community.document_loaders import PyPDFLoader
        try: docs = PyPDFLoader(path, extraction_mode="layout").load()      # layout mode keeps table rows on one line
        except TypeError: docs = PyPDFLoader(path).load()
        pages = [dict(page=int(d.metadata.get("page", i)) + 1, text=d.page_content or "", method="text") for i, d in enumerate(docs)]
    else:
        import pdfplumber
        with pdfplumber.open(path) as pdf:
            pages = [dict(page=i + 1, text=pg.extract_text() or "", method="text") for i, pg in enumerate(pdf.pages)]
    if any(len(p["text"].strip()) < CFG["MIN_TEXT_CHARS"] for p in pages):
        import pypdfium2 as pdfium, pytesseract
        pdf = pdfium.PdfDocument(path)
        try:
            for p in pages:
                if len(p["text"].strip()) < CFG["MIN_TEXT_CHARS"]:
                    p["text"] = pytesseract.image_to_string(pdf[p["page"] - 1].render(scale=2.5).to_pil().convert("L")); p["method"] = "ocr"
        finally: pdf.close()
    return pages

SECTION_MAP = [("ATTENDEES", "attendees"), ("PREVIOUS ACTION", "previous_actions"), ("PROGRESS", "progress"), ("MATERIAL", "materials"),
               ("CHANGE ORDER", "change_orders"), ("PAYMENT", "payments"), ("ACTION ITEMS", "new_actions"), ("KEY RISK", "key_risks"), ("NEXT MEETING", "next_meeting")]
HEADING = re.compile(r"^\s*(\d{1,2})\.\s+([A-Z][A-Z0-9 &/\-()]{3,})\s*$")
BULLET = re.compile(r"^\s*(?:\(cid:\d+\)|[^\w\s(]|[eo](?=\s))?\s*([A-Z][A-Za-z ]{2,22}?)(?:\s+Risk)?\s*(?:\((Red|Amber|Green|Closed)\))?\s*:\s+\S")
LABELS = {c.lower(): c for c in CATEGORIES}; LABELS["schedule status"] = "Schedule Status"
PID = r"([A-Z]{2,4}-\d{4}-\d{3})"

def canon_section(name):
    for key, c in SECTION_MAP:
        if key in name.upper(): return c
    return "other"

def parse_date(s):
    for f in ("%d %B %Y", "%d %b %Y"):
        try: return datetime.strptime(re.sub(r"\s+", " ", s.strip()), f).date().isoformat()
        except Exception: pass
    return None

def split_risk_bullets(text):
    out, cur = [], None
    for line in text.splitlines():
        m = BULLET.match(line)
        if m and m.group(1).strip().lower() in LABELS:
            if cur: out.append(cur)
            cur = dict(label=LABELS[m.group(1).strip().lower()], rag=m.group(2), body=re.sub(r"^[^:]*:\s*", "", line, count=1).strip())
        elif cur and line.strip(): cur["body"] += " " + line.strip()
    if cur: out.append(cur)
    for b in out: b["body"] = re.sub(r"\s+", " ", b["body"])
    return out

def norm(s): return re.sub(r"[^a-z0-9% ]+", " ", re.sub(r"\s+", " ", str(s).lower())).strip()
def cover(a, b):
    """fraction of text a that appears as one contiguous block in text b (0..1)"""
    a, b = norm(a), norm(b)
    if not a or not b: return 0.0
    m = difflib.SequenceMatcher(None, a, b, autojunk=False).find_longest_match(0, len(a), 0, len(b))
    return m.size / len(a)

def locate_page(rec, quote):
    """Page (1-based) of the source PDF where a quote appears: the traceable source reference."""
    q = norm(quote)[:60]
    if not q: return None
    for p in rec["pages"]:
        if q in norm(p["text"]): return p["page"]
    best = max(((cover(str(quote)[:200], p["text"]), p["page"]) for p in rec["pages"]), default=(0, None))
    return best[1] if best[0] >= 0.6 else None

def _num(s): return float(s.replace(",", ""))

def numeric_checks(rec):
    """Cross-section consistency checks (days behind, cumulative change %, remaining material quantities)."""
    T, S, issues = rec["full_text"], rec["sections"], []
    sec = lambda c: "\n".join(s["text"] for s in S if s["canonical"] == c)
    vals = sorted({int(v) for v in re.findall(r"(\d+)\s+(?:working\s+)?days?\s+behind", T, re.I)})
    if len(vals) > 1: issues.append(dict(kind="days_behind", detail=f"Schedule delay stated as different values: {vals}"))
    m = re.search(r"([\d.]+)%\s+of\s+(?:the\s+)?original contract value\s*\(\$([\d,.]+)\s*(M|million)?\)", T, re.I)
    co = sec("change_orders")
    if m and co:
        contract = _num(m.group(2)) * (1e6 if m.group(3) else 1)
        parts = re.split(r"(?=\b(?:P?CO)-\d+)", co)
        amts = [_num(a) for p in parts if p.startswith(("CO-", "PCO-")) and "rejected" not in p.lower() for a in re.findall(r"\+\$([\d,]+)", p)[:1]]
        if amts and contract:
            computed, stated = 100 * sum(amts) / contract, float(m.group(1))
            if abs(computed - stated) > 0.3:
                issues.append(dict(kind="cumulative_pct", detail=f"Stated {stated:.1f}% of contract but change orders sum to {computed:.1f}%"))
    mat = sec("materials")
    pats = [r"approximately\s+([\d,]+)\s+(\S+)\s+of\s+(.+?)\s+remaining", r"remaining\s+([\d,]+)\s+(?:tonnes|t|units|m)\s+of\s+([A-Za-z ]+?)(?:\s+from|\.|,|$)"]
    for i, pat in enumerate(pats):
        for g in re.finditer(pat, T, re.I):
            stated, name = _num(g.group(1)), (g.group(3) if i == 0 else g.group(2)).strip().lower()
            for line in mat.splitlines():
                l = re.sub(r"\bm(?:[³²]|[23](?![\d,]))", "m", line); key = name if i == 0 else name.split()[0]
                if key in l.lower():
                    rest = l[l.lower().find(key) + len(key):]
                    nums = [_num(x) for x in re.findall(r"(\d[\d,]*)\s*(?:tonnes|units|t|m)\b", re.sub(r"\([^)]*\)", "", rest))]
                    if len(nums) >= 3 and abs(nums[2] - stated) / max(nums[2], 1) > 0.01:
                        issues.append(dict(kind="material_qty", detail=f"Text says {stated:,.0f} remaining of '{name}', table says {nums[2]:,.0f}"))
                    break
    if rec.get("ocr"):
        for i in issues: i["detail"] += " [from OCR text - verify against the PDF]"
    return issues

ACTION_ID = re.compile(r"^\s*(A[Il1]-\d+)")      # tolerates OCR reading "AI-" as "Al-"
def parse_actions(rec):
    """Action items from the action tables, each with owner, due date, status and a source page."""
    out = []
    for s in rec["sections"]:
        if s["canonical"] not in ("previous_actions", "new_actions"): continue
        blocks, cur = [], None
        for line in s["text"].splitlines():
            m = ACTION_ID.match(line)
            if m:
                if cur: blocks.append(cur)
                cur = dict(id="AI" + m.group(1)[2:], text=line.strip())
            elif cur and line.strip() and not re.match(r"^\s*Item\s*#", line): cur["text"] += " " + line.strip()
        if cur: blocks.append(cur)
        for b in blocks:
            t = re.sub(r"\s+", " ", b["text"]); body = t[len(b["id"]) + 1:]
            owner = re.search(r"([A-Z]\. [A-Z][A-Za-z'\-]+(?: / [A-Z][.,] [A-Z][a-z]+)?(?: \([A-Za-z]+\))?)", body)
            due = re.search(r"(\d{1,2} [A-Z][a-z]{2} \d{4})", body)
            st = re.search(r"\d{4}\s+(OPEN\s*[-\u2013\u2014]\s*Overdue|OPEN|CLOSED|Open|Closed|Overdue)\b", body)
            sword = st.group(1).lower() if st else "open"
            status = "Overdue" if "overdue" in sword else ("Closed" if "closed" in sword else "Open")
            head = body[: owner.start()] if owner else body
            tail = body[st.end():] if st else ""                    # description text that wrapped onto the next line
            out.append(dict(id=b["id"], description=re.sub(r"\s+", " ", f"{head} {tail}").strip(" -,"), owner=owner.group(1) if owner else None,
                            due_date=parse_date(due.group(1)) if due else None, status=status, carried="carried forward" in t.lower(),
                            from_section=s["canonical"], page=locate_page(rec, b["id"] + " " + head[:30]) or s["page"], source=Path(rec["file"]).name))
    return out

## 5. Chunking (RecursiveCharacterTextSplitter) + ChromaDB indexing + retrieval

In [ ]:
def build_chunks(rec):
    """Split a parsed meeting into chunks (chunk_size=1000, overlap=200) carrying project / meeting / page / section metadata."""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=CFG["CHUNK_SIZE"], chunk_overlap=CFG["CHUNK_OVERLAP"])
    d_ord = int((rec["meeting_date"] or "0").replace("-", "") or 0)
    base = dict(doc_id=rec["doc_id"], project_no=rec["project_no"] or "UNKNOWN", meeting_no=int(rec["meeting_no"] or -1),
                date=rec["meeting_date"] or "", date_ord=d_ord, source=Path(rec["file"]).name)
    docs = []
    for s in rec["sections"]:
        if s["canonical"] == "key_risks":          # one chunk per risk bullet keeps each risk traceable
            pieces = [(f"{b['label']} Risk ({b['rag']}): {b['body']}" if b["rag"] else f"{b['label']}: {b['body']}") for b in split_risk_bullets(s["text"])] or [s["text"]]
        else: pieces = [s["text"]]
        for t in pieces:
            if t.strip(): docs.append(Document(page_content=t.strip(), metadata=dict(base, section=s["canonical"], page=int(locate_page(rec, t.strip()[:80]) or s["page"]))))
    chunks = splitter.split_documents(docs)
    for n, c in enumerate(chunks): c.metadata["chunk_id"] = f"{rec['doc_id']}::{n}"
    return chunks

def index_record(rec):
    chunks = build_chunks(rec)
    get_vectorstore().add_documents(chunks, ids=[c.metadata["chunk_id"] for c in chunks])      # upsert: re-uploads do not duplicate
    return len(chunks)

def _flt(**kw):
    items = [{k: v} for k, v in kw.items()]
    return items[0] if len(items) == 1 else {"$and": items}

def _as_chunk(doc): return dict(text=doc.page_content, meta=dict(doc.metadata))

def search(query, k=None, **filters):
    """Semantic search over the indexed minutes. filters: project_no=..., doc_id=..., section=..."""
    vs = get_vectorstore()
    return [_as_chunk(d) for d in vs.similarity_search(query, k=k or CFG["TOP_K"], filter=_flt(**filters) if filters else None)]

## 6. Graph state and nodes 1-5 (intake, extraction, metadata, chunk+index, retrieval)

In [ ]:
class S(TypedDict, total=False):
    files: List[str]; force: bool; docs: List[dict]; rejected: List[dict]; new_ids: List[str]
    records: Dict[str, dict]; evidence: Dict[str, List[dict]]; history: Dict[str, List[dict]]
    todo: List[str]; redo: List[str]; feedback: Dict[str, str]; retries: Dict[str, int]
    candidates: Dict[str, List[dict]]; assessed: Dict[str, List[dict]]; actions: Dict[str, List[dict]]
    register: Dict[str, list]; action_register: List[dict]; validated: Dict[str, List[dict]]; doc_issues: Dict[str, List[dict]]
    reviewed: List[dict]; reviewed_register: List[dict]; report: dict; log: Annotated[List[str], operator.add]

def node_intake(state: S):
    docs, rejected, seen, records, assessed, actions = [], [], {}, {}, {}, {}
    for f in state["files"]:
        p, why = Path(f), None
        if p.suffix.lower() != ".pdf": why = "not a PDF"
        elif not p.exists() or p.stat().st_size == 0: why = "missing or empty"
        elif p.stat().st_size > CFG["MAX_FILE_MB"] * 1e6: why = "file too large"
        else:
            h = hashlib.md5(p.read_bytes()).hexdigest()
            if h in seen: why = f"duplicate of {seen[h]}"
            else: seen[h] = p.name
        if why: rejected.append(dict(file=str(p), reason=why)); log_event("extraction_failure", p.stem, f"rejected at intake: {why}"); continue
        m = re.search(PID, p.stem) or re.search(PID, p.parent.name)
        cache, cached = Path(CFG["CACHE_DIR"]) / f"{p.stem}.json", False
        if cache.exists() and not state.get("force"):
            try:
                c = json.load(open(cache, encoding="utf-8"))
                if c.get("hash") == h:
                    records[p.stem], assessed[p.stem], actions[p.stem], cached = c["record"], c["assessed"], c["actions"], True
                    for a in assessed[p.stem]: a["score"] = score_of(a["likelihood"], a["impact"]); a["rag"] = "Closed" if a["stage"] == "closed" else rag_of(a["score"])      # re-score cached analyses with the current formula
            except Exception: pass
        docs.append(dict(doc_id=p.stem, file=str(p), project_hint=m.group(1) if m else None, hash=h, cached=cached))
    n_new = sum(not d["cached"] for d in docs)
    return {"docs": docs, "rejected": rejected, "records": records, "assessed": assessed, "actions": actions,
            "log": [f"[1 intake] {len(docs)} valid ({n_new} new, {len(docs)-n_new} already analysed), {len(rejected)} rejected"]}

def route_after_intake(state: S): return "pdf_extraction" if state["docs"] else END

def node_pdf_extraction(state: S):
    docs, log, n_ocr = [], [], 0
    for d in state["docs"]:
        if d["cached"]: docs.append(d); continue
        try: pages = load_pdf_pages(d["file"])
        except Exception as e:
            log_event("extraction_failure", d["doc_id"], f"{type(e).__name__}: {e}"); log.append(f"  ! cannot read {d['doc_id']}: {e}"); continue
        if any(p["method"] == "ocr" for p in pages): n_ocr += 1; log_event("ocr_used", d["doc_id"], "scanned pages read with OCR")
        if not any(len(p["text"].strip()) >= CFG["MIN_TEXT_CHARS"] for p in pages):
            log_event("extraction_failure", d["doc_id"], "no text could be extracted (is Tesseract installed?)"); log.append(f"  ! no text in {d['doc_id']}; skipped"); continue
        docs.append({**d, "pages": pages})
    return {"docs": docs, "log": [f"[2 extraction] {sum(not d['cached'] for d in docs)} new docs read ({n_ocr} via OCR)"] + log}

def node_metadata(state: S):
    records = dict(state.get("records", {}))
    for d in state["docs"]:
        if d["cached"] or "pages" not in d: continue
        sections, cur = [], dict(name="HEADER", canonical="header", text="", page=1)
        for p in d["pages"]:
            for line in p["text"].splitlines():
                m = HEADING.match(line)
                if m:
                    sections.append(cur); name = f"{m.group(1)}. {m.group(2).strip()}"
                    cur = dict(name=name, canonical=canon_section(name), text="", page=p["page"])
                else: cur["text"] += line + "\n"
        sections.append(cur); sections = [s for s in sections if s["text"].strip()]
        full = "\n".join(p["text"] for p in d["pages"])
        head = next((s["text"] for s in sections if s["canonical"] == "header"), full[:800])
        pn = re.search(PID, head) or re.search(PID, full)
        name = re.search(r"Project Name:\s*(.+?)(?:\s+Project No|\n|$)", head) or re.search(r"^\s*(.*?)\s+Project No:", head, re.M)
        mt, dt = re.search(r"Meeting\s*#\s*(\d+)", head), re.search(r"\bDate:\s*(\d{1,2}\s+[A-Za-z]+\s+\d{4})", head)
        records[d["doc_id"]] = dict(doc_id=d["doc_id"], file=d["file"], ocr=any(p["method"] == "ocr" for p in d["pages"]),
            project_no=pn.group(1) if pn else d["project_hint"], project_name=name.group(1).strip() if name else None,
            meeting_no=int(mt.group(1)) if mt else None, meeting_date=parse_date(dt.group(1)) if dt else None,
            sections=sections, pages=d["pages"], full_text=full, n_pages=len(d["pages"]))
    miss = [k for k, r in records.items() if not r["project_no"] or not r["meeting_date"]]
    for k in miss: log_event("extraction_failure", k, "project number or meeting date could not be parsed")
    return {"records": records, "log": [f"[3 metadata] {len(records)} records" + (f"; incomplete metadata: {miss[:5]}" if miss else "")]}

def node_chunk_index(state: S):
    n = 0
    for d in state["docs"]:
        if not d["cached"] and d["doc_id"] in state["records"]: n += index_record(state["records"][d["doc_id"]])
    return {"log": [f"[4 chunk+index] {n} chunks stored in ChromaDB ({CFG['_embedder']} embeddings); collection size {get_vectorstore()._collection.count()}"]}

QUERIES = ["risk exposure concern problem", "cost increase price escalation change order", "schedule behind delay late delivery",
           "overdue open action item", "supply shortage supplier dependency", "safety quality environmental permit regulator"]
def node_retrieval(state: S):
    vs, evidence, history = get_vectorstore(), {}, {}
    new_ids = [d["doc_id"] for d in state["docs"] if not d["cached"] and d["doc_id"] in state["records"]]
    for rid in new_ids:
        r = state["records"][rid]; got = {}
        res = vs.get(where=_flt(doc_id=rid), include=["documents", "metadatas"])
        for t, m in zip(res["documents"], res["metadatas"]):
            if m["section"] in ("key_risks", "change_orders", "progress"): got[m["chunk_id"]] = dict(text=t, meta=m)
        for q in QUERIES:
            for c in search(q, CFG["TOP_K"], doc_id=rid): got.setdefault(c["meta"]["chunk_id"], c)
        evidence[rid] = sorted(got.values(), key=lambda c: (c["meta"]["page"], c["meta"]["chunk_id"]))[:20]
        d_ord = int((r["meeting_date"] or "0").replace("-", "") or 0)
        try:
            hs = vs.similarity_search("risk status update delay cost supplier", k=6, filter={"$and": [{"project_no": r["project_no"] or "UNKNOWN"}, {"section": "key_risks"}, {"date_ord": {"$lt": d_ord}}]})
            history[rid] = [_as_chunk(h) for h in hs]
        except Exception: history[rid] = []
    return {"evidence": evidence, "history": history, "new_ids": new_ids, "todo": list(new_ids), "feedback": {}, "retries": {}, "candidates": {},
            "log": [f"[5 retrieval] evidence passages for {len(new_ids)} docs (+ earlier meetings of the same project as context)"]}

## 7. Nodes 6-7: risk & action extraction (LLM + structured schema) and classification

In [ ]:
class RiskCandidate(BaseModel):
    title: str
    category: str = "Schedule"
    description: str = ""
    signals: List[str] = []
    cause: str = ""
    consequence: str = ""
    owner: Optional[str] = None
    due_date: Optional[str] = None
    actions: List[str] = []
    stage: str = "open"
    evidence_quote: str = ""
    source_section: str = ""
    page: Optional[int] = None
    confidence: float = 0.7

EXTRACT_SYS = f"""You are a construction project risk analyst. From meeting-minute passages, extract ONLY risks actually discussed in this meeting.
Return JSON: {{"risks":[{{"title":short noun phrase,"category":one of {CATEGORIES},"description":str,"signals":[str],"cause":str,"consequence":str,
"owner":name from the action items or null,"due_date":ISO date or null,"actions":[str],"stage":one of {STAGES},"evidence_quote":text copied VERBATIM from the passages (max 250 chars),
"source_section":str,"page":int,"confidence":0-1}}]}}.
Stages: raised=newly flagged; escalated=getting worse/now impacting work; mitigating=being resolved; closed=resolved; open=ongoing, no change.
Rules: never invent owners, dates, or numbers. Skip administrative items. Overall schedule status counts as a Schedule risk only if a cause or threat is described."""

def _stage_rule(body, rag):
    b = body.lower()
    if rag == "Closed" or "risk closed" in b or re.search(r"\bclosed\b", b): return "closed"
    if re.search(r"resolved|agreed|mobiliz|recover|improving|reduced|implemented|installed|delivered|completed|under way|underway|secured|resuming|being rebuilt|are in place", b): return "mitigating"
    if re.search(r"\bnow\b|second|persist|unresolved|not yet|on hold|confirmed|required|increased|formally|exposure|constrain|delaying|holding up|has missed|issued an order", b): return "escalated"
    return "raised"

def _find_owner(rec, title):
    """Owner, due date and wording of the action item that best matches the risk title (word-stem overlap)."""
    stems = lambda s: {w.rstrip("s")[:6] for w in norm(s).split() if len(w) > 4} - {"delay", "risk", "issue"}
    want, best = stems(title), (0, None)
    for a in parse_actions(rec):
        n = len(want & stems(a["description"]))
        if n > best[0]: best = (n, a)
    a = best[1]
    return (a["owner"], a["due_date"], re.sub(r"\s*-\s*\([^)]*\)\s*(Overdue|Open|Closed)?\s*$", "", a["description"]).strip()) if a else (None, None, None)

def _rule_extract(rec, ev):
    out = []
    for e in ev:
        if e["meta"]["section"] != "key_risks": continue
        for b in split_risk_bullets(e["text"]):
            if b["label"] == "Schedule Status": continue
            first = re.split(r"(?<=[a-z0-9\)])\.\s", b["body"], maxsplit=1)
            title, rest = first[0][:120], (first[1] if len(first) > 1 else "")
            owner, due, act = _find_owner(rec, title)
            out.append(dict(title=title, category=b["label"], description=rest or b["body"], owner=owner, due_date=due, actions=[act] if act else [], stage=_stage_rule(b["body"], b["rag"]),
                            evidence_quote=b["body"][:250], source_section="key_risks", confidence=0.75 if not rec["ocr"] else 0.6))
    return out

def _fix_category(c):
    for k in CATEGORIES:
        if str(c).strip().lower() == k.lower(): return k
    best = difflib.get_close_matches(str(c).title(), CATEGORIES, n=1, cutoff=0.5)
    return best[0] if best else "Schedule"

def node_extract_risks(state: S):
    cands, feedback, n_llm = dict(state.get("candidates", {})), state.get("feedback", {}), 0
    actions = dict(state.get("actions", {}))
    for rid in state["todo"]:
        rec, ev = state["records"][rid], state["evidence"][rid]
        passages = "\n\n".join(f"[{e['meta']['section']} | page {e['meta']['page']}]\n{e['text']}" for e in ev)
        hist = "\n".join(f"- (meeting {h['meta']['meeting_no']}) {h['text'][:200]}" for h in state["history"][rid]) or "none"
        user = (f"Project {rec['project_no']} meeting #{rec['meeting_no']} on {rec['meeting_date']}.\nRisks from earlier meetings (context only):\n{hist}\n\n"
                f"PASSAGES FROM THIS MEETING:\n{passages}\n" + (f"\nREVIEWER FEEDBACK (fix these problems): {feedback[rid]}\n" if rid in feedback else ""))
        res, raw = llm_json(EXTRACT_SYS, user, rid), None
        if res and isinstance(res.get("risks"), list): raw, n_llm = res["risks"], n_llm + 1
        if raw is None: raw = _rule_extract(rec, ev)
        good = []
        for r in raw:
            try:
                c = RiskCandidate(**{k: v for k, v in r.items() if k in RiskCandidate.model_fields and v is not None}).model_dump()
                c["category"] = _fix_category(c["category"]); c["stage"] = c["stage"] if c["stage"] in STAGES else "open"
                found = locate_page(rec, c["evidence_quote"])
                c.update(page=found or c.get("page"), source=Path(rec["file"]).name, doc_id=rid, meeting_no=rec["meeting_no"], quote_located=bool(found))
                good.append(c)
            except Exception as e: log_event("validation_failure", rid, f"candidate rejected by schema: {e}")
        cands[rid] = good
        actions[rid] = parse_actions(rec)
        if not actions[rid] and any(s["canonical"] in ("previous_actions", "new_actions") and re.search(r"Item\s*#", s["text"]) for s in rec["sections"]):
            log_event("extraction_failure", rid, "action-item tables found but no action items could be parsed")
    return {"candidates": cands, "actions": actions, "redo": list(state["todo"]), "todo": [],
            "log": [f"[6 extraction] {sum(len(cands[r]) for r in state['todo'])} candidate risks and {sum(len(actions[r]) for r in state['todo'])} action items ({n_llm} docs via LLM)"]}

CLASSIFY_SYS = f"""You assess construction project risks using this rubric:\n{RUBRIC}
Return JSON {{"assessments":[{{"index":int,"category":one of {CATEGORIES},"likelihood":1-5,"impact":1-5,"type":"Risk" or "Issue","response_strategy":one of {STRATEGIES},"rationale":"one sentence citing the evidence"}}]}}.
type: Issue = it has already happened or is certain (e.g. "deliveries are now 10 days late"); Risk = it may still happen (e.g. "may slip by up to 10 days").
response_strategy (PMI threat responses) must come ONLY from the actions listed for the risk: Avoid = remove the cause or change the plan; Transfer = shift the impact to a third party (insurance, contract, supplier);
Mitigate = reduce the likelihood or impact; Accept = knowingly carry it (monitor, contingency reserve); Escalate = refer it above the project team. Use "Not stated" when no action is listed."""

def strategy_of(text):
    """PMI response strategy from the wording of an action (rule-based fallback; approximate, the reviewer can change it)."""
    t = (text or "").lower()
    if not t.strip(): return "Not stated"
    if re.search(r"\bescalat(e|es|ed|ing)\b|refer (this )?to (the )?(sponsor|owner|executive|client)|raise (this )?with (the )?(sponsor|owner|executive)", t): return "Escalate"
    if re.search(r"insur|indemn|transfer|warrant|\bbond\b|back-to-back|liquidated", t): return "Transfer"
    if re.search(r"avoid|eliminat|re-?sequenc|re-?rout|re-?design|replace the|cancel|change the (design|method|scope)|alternative (route|method|design)", t): return "Avoid"
    if re.search(r"accept|absorb|tolerate|no action|monitor only|contingency reserve|\breserve\b", t): return "Accept"
    return "Mitigate"

def type_of(c, L):
    """Issue = already happening (PMI risk vs issue); Risk = may still happen. Rule-based fallback."""
    t = f"{c.get('description', '')} {c.get('evidence_quote', '')}".lower()
    happened = re.search(r"\b(now|already|has been|have been|has missed|confirmed|encountered|exposed|lost|failed|occurred|delaying|holding up|overdue|behind)\b|\bis (delayed|late)", t)
    return "Issue" if L >= 5 or c.get("stage") == "escalated" or happened else "Risk"

IMPACT_BASE = dict(Safety=5, Contractual=4, Regulatory=4, Cost=4, Technical=4, Environmental=4, Schedule=3, Quality=3, Resource=3, Stakeholder=2, **{"Supply Chain": 4})

def node_classify(state: S):
    assessed = dict(state.get("assessed", {}))
    for rid in state["redo"]:
        cs = state["candidates"][rid]
        res = llm_json(CLASSIFY_SYS, json.dumps([dict(index=i, title=c["title"], category=c["category"], stage=c["stage"], description=c["description"],
                                                      evidence=c["evidence_quote"], actions=c.get("actions") or []) for i, c in enumerate(cs)]), rid) if cs else None
        by = {a.get("index"): a for a in (res or {}).get("assessments", [])}
        out = []
        for i, c in enumerate(cs):
            c, a = dict(c), by.get(i)
            if a:
                c["category"] = _fix_category(a.get("category", c["category"])); L, I, why = int(a.get("likelihood", 3)), int(a.get("impact", 3)), a.get("rationale", "")
            else:
                L = dict(raised=3, escalated=4, mitigating=2, closed=1, open=3)[c["stage"]]
                I = 1 if c["stage"] == "closed" else IMPACT_BASE.get(c["category"], 3) - (1 if c["stage"] == "mitigating" else 0)
                why = f"Rule-based: stage '{c['stage']}' in category '{c['category']}' (rubric: probability x impact)."
            L, I = max(1, min(5, L)), max(1, min(5, I))
            acts_txt = "; ".join(str(x) for x in (c.get("actions") or [])); ai = a or {}
            sv, tv = str(ai.get("response_strategy", "")).strip().capitalize(), str(ai.get("type", "")).strip().capitalize()
            c["response_action"], c["response_strategy"], c["type"] = acts_txt[:300], (sv if sv in STRATEGIES else strategy_of(acts_txt)), (tv if tv in TYPES else type_of(c, L))
            c.update(likelihood=L, impact=I, score=score_of(L, I), rag="Closed" if c["stage"] == "closed" else rag_of(score_of(L, I)), rationale=why); out.append(c)
        assessed[rid] = out
    return {"assessed": assessed, "log": [f"[7 classification] {sum(len(assessed[r]) for r in state['redo'])} risks assessed with the likelihood x impact rubric"]}

## 8. Node 8: historical matching (cross-meeting register + action-item status) and Node 9: quality check

In [ ]:
_match_cache = {}
def _llm_match(c, entries):
    key = (c["title"], tuple(e["reg_id"] for e in entries))
    if key in _match_cache: return _match_cache[key]
    opts = "\n".join(f"{e['reg_id']}: {e['title']} -- {e['texts'][-1][-200:]}" for e in entries)
    res = llm_json('Decide whether a newly reported construction risk is the SAME underlying risk as one in the register. Return JSON {"match": "<reg_id or null>"}.',
                   f"New risk: {c['title']} -- {c['description']} {c['evidence_quote']}\n\nRegister:\n{opts}")
    m = (res or {}).get("match"); _match_cache[key] = m if m in {e["reg_id"] for e in entries} else None
    return _match_cache[key]

def _iso(x):
    """Date text -> 'YYYY-MM-DD' (accepts ISO or '12 Jun 2025'); None if unreadable."""
    if not x: return None
    s = str(x).strip()
    try: return datetime.fromisoformat(s[:10]).date().isoformat()
    except Exception: return parse_date(s)

def _days(a, b):
    """Whole days from date a to date b (positive when b is later). None if either date is missing or unreadable."""
    a, b = _iso(a), _iso(b)
    return None if not a or not b else (datetime.fromisoformat(b) - datetime.fromisoformat(a)).days

def meeting_clock(records):
    """Per project: date and number of the LATEST uploaded meeting. Overdue checks are made against this 'as of' date."""
    asof, latest = {}, {}
    for r in records.values():
        p = r.get("project_no") or "UNKNOWN"
        if r.get("meeting_date") and r["meeting_date"] >= asof.get(p, ""): asof[p] = r["meeting_date"]
        if r.get("meeting_no") is not None: latest[p] = max(latest.get(p, 0), r["meeting_no"])
    return asof, latest

def build_action_register(records, actions):
    """Cross-meeting action tracking. Overdue is decided from the due date vs the latest meeting date, not only from the minutes' own wording."""
    asof, latest_no = meeting_clock(records)
    rows = {}
    for rid in sorted(actions, key=lambda d: (records[d]["project_no"] or "", records[d]["meeting_date"] or "", records[d]["meeting_no"] or 0)):
        r = records[rid]
        for a in actions[rid]:
            e = rows.setdefault((r["project_no"], a["id"]), dict(project_no=r["project_no"], action_id=a["id"], description=a["description"], owner=a["owner"],
                                  first_meeting=r["meeting_no"], history=[]))
            e["history"].append(dict(meeting_no=r["meeting_no"], date=r["meeting_date"], status=a["status"], due_date=a["due_date"], page=a["page"], source=a["source"]))
            e.update(last_meeting=r["meeting_no"], status=a["status"], due_date=a["due_date"], owner=a["owner"] or e["owner"], page=a["page"], source=a["source"])
    out = []
    for e in rows.values():
        h, p = e["history"], e["project_no"] or "UNKNOWN"
        e["status_updates"] = " -> ".join(f"M{x['meeting_no']}: {x['status']}" for x in h)
        e["carried_count"] = sum(1 for x in h[1:] if x["status"] != "Closed"); e["n_mentions"] = len(h)
        e["status_in_minutes"] = e["status"]
        late = _days(e["due_date"], asof.get(p)) if e["status"] != "Closed" else None      # > 0 means the due date had already passed at the latest meeting
        e["days_overdue"] = late if late and late > 0 else 0
        if e["days_overdue"]: e["status"] = "Overdue"
        prev = h[-1] if e["last_meeting"] < latest_no.get(p, 0) else (h[-2] if len(h) > 1 else None)   # state at the previous meeting: was it already overdue then?
        was = bool(prev) and (prev["status"] == "Overdue" or (_days(prev["due_date"], prev["date"]) or 0) > 0)
        flags = []
        if e["status"] == "Overdue": flags.append(f"OVERDUE by {e['days_overdue']} days" if e["days_overdue"] else "OVERDUE (per the minutes)")
        if e["status"] == "Overdue" and not was: flags.append("NEWLY overdue")
        if e["status"] != "Closed" and late is not None and -7 <= late <= 0: flags.append(f"Due in {-late} days" if late else "Due today")
        if e["status"] != "Closed" and e["last_meeting"] is not None and e["last_meeting"] < latest_no.get(p, 0): flags.append("Not mentioned in latest meeting")
        moves = len({x["due_date"] for x in h if x["due_date"]}) - 1
        if moves > 0: flags.append(f"Due date moved {moves}x")
        if e["carried_count"] >= 3 and e["status"] != "Closed": flags.append(f"Carried forward {e['carried_count']}x")
        e["attention"] = "; ".join(flags)
        out.append({k: v for k, v in e.items() if k != "history"})
    out.sort(key=lambda e: (e["status"] != "Overdue", -e["days_overdue"], e["status"] == "Closed", e["project_no"] or "", e["action_id"]))
    return out

def node_match(state: S):
    recs, thr = state["records"], CFG["_thr"]
    reg, assessed = {}, {}
    order = sorted(state["assessed"], key=lambda d: (recs[d]["project_no"] or "", recs[d]["meeting_date"] or "", recs[d]["meeting_no"] or 0))
    for rid in order:
        r = recs[rid]; entries, used = reg.setdefault(r["project_no"] or "UNKNOWN", []), set(); assessed[rid] = []
        for c in state["assessed"][rid]:
            c = dict(c); text = f"{c['category']} {c['title']} {c['description']} {c['evidence_quote']}"
            pool = [e for e in entries if e["category"] == c["category"] and e["status"] != "closed" and e["reg_id"] not in used]
            dup = [e for e in entries if e["reg_id"] in used and e["category"] == c["category"]]
            best, sim, is_dup = None, 0.0, False
            def sims_to(es): return np.array([max(float(embed_matrix([text])[0] @ v) for v in embed_matrix(e["texts"])) for e in es]) if es else np.array([])
            if dup:
                ds = sims_to(dup)
                if ds.max() >= 0.8: best, sim, is_dup = dup[int(ds.argmax())], float(ds.max()), True
            if pool and not is_dup:
                sims = sims_to(pool); j = int(np.argmax(sims)); best, sim = pool[j], float(sims[j])
                if sim < thr:
                    near = [pool[k] for k in np.argsort(-sims)[:3] if sims[k] > thr * 0.5]
                    m = _llm_match(c, near) if near else None
                    best = next((e for e in near if e["reg_id"] == m), None)
            c["match_similarity"] = round(sim, 3)
            if best is None:
                e = dict(reg_id=f"{(r['project_no'] or 'UNK')}-R{len(entries)+1:02d}", title=c["title"], category=c["category"], first_meeting=r["meeting_no"], first_date=r["meeting_date"],
                         last_meeting=r["meeting_no"], last_date=r["meeting_date"], status=c["stage"], score=c["score"], type=c.get("type", "Risk"), response_strategy=c.get("response_strategy", "Not stated"), response_action=c.get("response_action", ""), texts=[text], history=[])
                entries.append(e); c.update(reg_id=e["reg_id"], is_new=True, status_change=None, duplicate=False); best = e
            else: c.update(reg_id=best["reg_id"], is_new=False, duplicate=is_dup, status_change=f"{best['status']} -> {c['stage']}" if best["status"] != c["stage"] else None)
            used.add(best["reg_id"])
            if best.get("type") == "Issue": c["type"] = "Issue"          # once a risk has happened it stays an issue (it remains in the register)
            if not c.get("response_action") and best.get("response_action"): c["response_action"], c["response_strategy"] = best["response_action"], best.get("response_strategy", "Not stated")      # keep the last stated response
            if not c["duplicate"] and not any(h["doc_id"] == rid for h in best["history"]):
                best["history"].append(dict(doc_id=rid, meeting_no=r["meeting_no"], date=r["meeting_date"], stage=c["stage"], score=c["score"], rag=c["rag"], title_as_written=c["title"], page=c.get("page")))
                best.update(last_meeting=r["meeting_no"], last_date=r["meeting_date"], status=c["stage"], score=c["score"], type=c.get("type", "Risk"), response_strategy=c.get("response_strategy", "Not stated"), response_action=c.get("response_action", ""), texts=(best["texts"] + [text])[-5:])
            assessed[rid].append(c)
    arr = build_action_register(recs, state.get("actions", {}))
    return {"assessed": assessed, "register": reg, "action_register": arr,
            "log": [f"[8 matching] register holds {sum(len(v) for v in reg.values())} distinct risks across {len(reg)} project(s); {len(arr)} tracked action items ({sum(a['status'] == 'Overdue' for a in arr)} overdue)"]}

REQUIRED = ["title", "category", "description", "evidence_quote"]
def node_quality(state: S):
    recs, validated, doc_issues, todo, fb, new = state["records"], {}, {}, [], {}, set(state.get("new_ids", []))
    retries = dict(state.get("retries", {}))
    for rid, cs in state["assessed"].items():
        rec, out = recs[rid], []
        for c in cs:
            c, iss = dict(c), []
            if [f for f in REQUIRED if not c.get(f)]: iss.append("missing_required_field")
            if c.get("evidence_quote") and cover(c["evidence_quote"], rec["full_text"]) < 0.7: iss.append("unsupported_evidence")
            if c.get("confidence", 1) < CFG["CONF_MIN"]: iss.append("low_confidence")
            if c.get("duplicate"): iss.append("duplicate_in_document")
            if iss and rid in new:
                for i in iss: log_event("unsupported_claim" if i == "unsupported_evidence" else i, rid, f"{c['title'][:80]}", reg_id=c.get("reg_id"))
            c["issues"], c["needs_review"] = iss, bool(iss); out.append(c)
        validated[rid] = out; doc_issues[rid] = numeric_checks(rec)
        if rid in new:
            for i in doc_issues[rid]: log_event("inconsistency", rid, i["detail"], issue=i["kind"])
        hard = sum(1 for c in out if {"unsupported_evidence", "missing_required_field"} & set(c["issues"]))
        if rid in new and cs and hard / len(cs) > 0.3 and retries.get(rid, 0) < CFG["MAX_RETRIES"]:
            todo.append(rid); retries[rid] = retries.get(rid, 0) + 1
            fb[rid] = "Some evidence_quote values were not verbatim or required fields were empty. Copy quotes exactly from the passages and fill every required field."
    flagged = sum(1 for rid in new for c in validated.get(rid, []) if c["needs_review"]); nums = sum(len(doc_issues[r]) for r in new if r in doc_issues)
    return {"validated": validated, "doc_issues": doc_issues, "todo": todo, "feedback": fb, "retries": retries,
            "log": [f"[9 quality] {flagged} risks flagged, {nums} cross-section inconsistencies, {len(todo)} docs sent back for re-extraction"]}

## 9. Node 9b: human-in-the-loop review (the graph PAUSES here until the project manager submits decisions)
The reviewer sees existing vs proposed values with the reason and the source page, can edit / approve / reject, and items needing clarification stay visibly unresolved.
Decisions and corrections are remembered (so already-reviewed items are not asked again) and written to the audit log.

In [ ]:
REVIEW_FIELDS = ["stage", "likelihood", "impact", "owner", "due_date", "type", "response_strategy"]
DECISIONS = ["Pending", "Approved", "Rejected", "Needs clarification"]
REVIEWS_PATH = Path(CFG["CACHE_DIR"]) / "reviews.json"
def load_reviews():
    try: return json.load(open(REVIEWS_PATH, encoding="utf-8"))
    except Exception: return {}
REVIEWS: Dict[str, dict] = load_reviews()

def _clean(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if hasattr(v, "item"): v = v.item()
    return v

REGISTER_COLS = ["reg_id", "project_no", "title", "description", "category", "type", "likelihood", "impact", "score", "rationale", "rag",
                 "stage", "current_status", "first_identified", "risk_age_days", "update_flag", "change_since_last", "overdue_flag", "owner", "response_strategy", "response_action", "due_date",
                 "evidence", "source", "page", "review_state", "unresolved_items", "comment"]
def order_register(df):
    """Put the register columns in the agreed order (description after title, rationale after score); any extra columns go last."""
    cols = [c for c in REGISTER_COLS if c in df.columns]
    return df[cols + [c for c in df.columns if c not in cols]]

def _risk_overdue(r, asof):
    """Open risk whose own due date (as approved or edited by the reviewer) is already past at the latest meeting."""
    if r.get("stage") == "closed" or not r.get("due_date"): return ""
    d = _days(r["due_date"], asof.get(r.get("project_no") or "UNKNOWN"))
    return f"OVERDUE by {d} days" if d and d > 0 else ""

def build_frames(state):
    recs, rows = state["records"], []
    asof, latest_no = meeting_clock(recs)
    for rid, cs in state.get("validated", {}).items():
        r = recs[rid]
        for c in cs:
            rows.append(dict(doc_id=rid, project_no=r["project_no"], meeting_no=r["meeting_no"], date=r["meeting_date"], reg_id=c.get("reg_id"), title=c["title"],
                             description=" ".join(str(c.get("description") or "").split())[:400],
                             type=c.get("type", "Risk"), response_strategy=c.get("response_strategy", "Not stated"), response_action=c.get("response_action", ""),
                             category=c["category"], stage=c["stage"], likelihood=c["likelihood"], impact=c["impact"], score=c["score"], rag=c["rag"], owner=c.get("owner"),
                             due_date=c.get("due_date"), status_change=c.get("status_change"), needs_review=c["needs_review"], issues=",".join(c["issues"]),
                             rationale=c["rationale"], evidence=c["evidence_quote"], source=c.get("source"), page=c.get("page"), confidence=c.get("confidence"), ocr=r["ocr"]))
    findings = pd.DataFrame(rows)
    reg = []
    for p, es in state.get("register", {}).items():
        for e in es:
            h = e["history"]; last, prev = h[-1], (h[-2] if len(h) > 1 else None)
            closed, now = e["status"] == "closed", latest_no.get(p, e["last_meeting"])
            age = _days(e["first_date"], e["last_date"] if closed else asof.get(p))       # closed risks stop ageing at the meeting that closed them
            if e["last_meeting"] < now: flag, why = ("Closed", "") if closed else ("Not discussed in latest meeting", f"Last discussed in M{e['last_meeting']}; latest meeting is M{now}")
            elif prev is None: flag, why = "New", f"First raised in M{e['last_meeting']}"
            else:
                d, bits = (last["score"] or 0) - (prev["score"] or 0), []
                if d: bits.append(f"score {prev['score']} -> {last['score']}")
                if last["stage"] != prev["stage"]: bits.append(f"stage {prev['stage']} -> {last['stage']}")
                flag = "Closed this meeting" if closed and prev["stage"] != "closed" else "Escalated" if d > 0 else "Improved" if d < 0 else "Stage changed" if bits else "No change"
                why = ", ".join(bits)
            reg.append(dict(project_no=p, reg_id=e["reg_id"], title=e["title"], category=e["category"], status=e["status"], score=e["score"],
                            first_identified=e["first_date"], risk_age_days=age, current_status="Closed" if closed else "Open",
                            update_flag=flag, change_since_last=why, last_meeting=e["last_meeting"], n_mentions=len(h),
                            response_action=e.get("response_action", "")))
    inc = [dict(doc_id=d, project_no=recs[d]["project_no"], meeting_no=recs[d]["meeting_no"], **i) for d, v in state.get("doc_issues", {}).items() for i in v]
    return findings, pd.DataFrame(reg), pd.DataFrame(inc), pd.DataFrame(state.get("action_register", []))

def _rkey(doc_id, title): return f"{doc_id}|{norm(title)[:80]}"

def build_proposals(findings, inc):
    """One proposal per new risk, per change vs the previous meeting, and per item flagged by the quality check."""
    if findings.empty: return []
    notes = {}
    for _, r in inc.iterrows(): notes.setdefault(r["doc_id"], []).append(r["detail"])
    out, f = [], findings.sort_values(["project_no", "reg_id", "meeting_no"])
    for reg_id, g in f.groupby("reg_id", sort=False):
        prev = None
        for _, r in g.iterrows():
            prop = {k: _clean(r[k]) for k in REVIEW_FIELDS}; old = None if prev is None else {k: _clean(prev[k]) for k in REVIEW_FIELDS}
            changed = [k for k in REVIEW_FIELDS if old is None or old[k] != prop[k]]; unclear = bool(r["needs_review"])
            if changed or unclear:
                p = dict(pid=len(out) + 1, key=_rkey(r["doc_id"], r["title"]), project_no=r["project_no"], reg_id=reg_id, meeting_no=int(r["meeting_no"]), doc_id=r["doc_id"],
                         title=r["title"], description=r["description"], category=r["category"], kind="new risk" if old is None else "update", existing=old, proposed=prop, original=dict(prop), changed=changed,
                         evidence=r["evidence"], rationale=r["rationale"], issues=r["issues"], source=_clean(r["source"]), page=_clean(r["page"]),
                         doc_notes=notes.get(r["doc_id"], []), decision="Needs clarification" if unclear else "Pending", comment="")
                rv = REVIEWS.get(p["key"])
                if rv: p["decision"], p["comment"] = rv["decision"], rv.get("comment", ""); p["proposed"].update(rv.get("edits", {}))
                out.append(p)
            prev = r
    return out

def summary(props): return {d: sum(p["decision"] == d for p in props) for d in DECISIONS}

def record_decisions(props):
    """Remember decisions + log corrections (audit trail for 'user corrections')."""
    for p in props:
        if p["decision"] == "Pending": continue
        edits = {k: v for k, v in p["proposed"].items() if v != p["original"].get(k)}
        prev = REVIEWS.get(p["key"], {})
        if prev.get("decision") != p["decision"] or prev.get("edits") != edits or prev.get("comment") != p["comment"]:
            log_event("review_decision", p["doc_id"], f"{p['title'][:70]} -> {p['decision']}", reg_id=p["reg_id"], comment=p["comment"])
            for k, v in edits.items(): log_event("user_correction", p["doc_id"], f"{p['title'][:60]}: {k} {p['original'].get(k)!r} -> {v!r}", reg_id=p["reg_id"], field=k)
        REVIEWS[p["key"]] = dict(decision=p["decision"], comment=p["comment"], edits=edits, reviewed_at=datetime.now().isoformat(timespec="seconds"))
    json.dump(REVIEWS, open(REVIEWS_PATH, "w", encoding="utf-8"), indent=1)

def apply_decisions(props):
    """Reviewed register: approved values only; rejected items are dropped; unresolved items stay visibly unresolved."""
    cur = {}
    for p in sorted(props, key=lambda p: (p["reg_id"], p["meeting_no"])):
        row = cur.setdefault(p["reg_id"], dict(reg_id=p["reg_id"], project_no=p["project_no"], approved=False, rejected=0, unresolved=0, comment="", last=None))
        row["last"] = p
        if not row.get("description"): row["description"] = p.get("description", "")      # describe the risk as first identified; later updates go in the rationale
        if p["decision"] == "Approved":
            row.update(p["proposed"]); row.update(title=p["title"], category=p["category"], approved=True, rationale=p["rationale"], evidence=p["evidence"], source=p["source"], page=p["page"], comment=p["comment"] or row["comment"])
        elif p["decision"] == "Rejected": row["rejected"] += 1
        else: row["unresolved"] += 1
    rows = []
    for r in cur.values():
        lp = r.pop("last")
        if not r["approved"]:
            if r["unresolved"] == 0 and r["rejected"] > 0: continue
            r.update(lp["proposed"]); r.update(title=lp["title"], category=lp["category"], rationale=lp["rationale"], evidence=lp["evidence"], source=lp["source"], page=lp["page"])
            r["review_state"] = f"UNRESOLVED ({r['unresolved']} open)" if r["unresolved"] else "Unreviewed"
        else: r["review_state"] = "Approved" + (f" + UNRESOLVED ({r['unresolved']} open)" if r["unresolved"] else "")
        r["score"] = score_of(r["likelihood"], r["impact"]); r["rag"] = "Closed" if r.get("stage") == "closed" else rag_of(r["score"]); r["unresolved_items"] = r.pop("unresolved")
        rows.append({k: v for k, v in r.items() if k not in ("approved", "rejected")})
    df = pd.DataFrame(rows)
    log = [dict(pid=p["pid"], reg_id=p["reg_id"], meeting_no=p["meeting_no"], decision=p["decision"], comment=p["comment"], **{f"proposed_{k}": v for k, v in p["proposed"].items()}) for p in props if p["decision"] != "Pending"]
    pd.DataFrame(log).to_csv(Path(CFG["OUT_DIR"]) / "review_decisions.csv", index=False)
    return df

def node_human_review(state: S):
    findings, reg, inc, _ = build_frames(state)
    props = build_proposals(findings, inc)
    pending = [p for p in props if p["decision"] in ("Pending", "Needs clarification")]
    decided = interrupt({"message": "Review suggested risk changes", "proposals": props}) if (CFG["HUMAN_REVIEW"] and pending) else props   # <- graph pauses here
    record_decisions(decided)
    df = apply_decisions(decided)
    if not df.empty and not reg.empty:
        df = df.merge(reg[["reg_id", "first_identified", "risk_age_days", "update_flag", "change_since_last", "response_action"]], on="reg_id", how="left")
        df["risk_age_days"] = pd.to_numeric(df["risk_age_days"], errors="coerce").astype("Int64")
        df["current_status"] = df["stage"].map(lambda s: "Closed" if s == "closed" else "Open")
        asof, _ = meeting_clock(state["records"])
        df["overdue_flag"] = df.apply(lambda r: _risk_overdue(r, asof), axis=1)
        df = order_register(df)
    return {"reviewed": decided, "reviewed_register": json.loads(df.to_json(orient="records")),
            "log": [f"[9b human review] decisions {summary(decided)}; reviewed register has {len(df)} risks"]}

## 10. Node 10: report and export (CSV + PDF with reasons, evidence and source pages)

In [ ]:
TINTS = [("Red", "#f8b4b4"), ("Amber", "#fde3a7"), ("Green", "#bfe3c0"), ("Closed", "#d9d9d9"), ("Overdue", "#f8b4b4"), ("OVERDUE", "#f8b4b4"),
         ("Escalated", "#f8b4b4"), ("NEWLY", "#f8b4b4"), ("New", "#fde3a7"), ("Due in", "#fde3a7"), ("Due today", "#fde3a7"), ("Stage changed", "#fde3a7"),
         ("Improved", "#bfe3c0"), ("Not discussed", "#e5e5e5")]      # colour key shared by the PDF and the Excel export
def tint_of(v):
    s = str(v if v is not None and v == v else "")
    return next((c for k, c in TINTS if s.startswith(k)), None)

def make_pdf(path, reviewed, actions, inc, audit, summary_line):
    from reportlab.lib.pagesizes import letter, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from xml.sax.saxutils import escape as esc
    st = getSampleStyleSheet(); cell = ParagraphStyle("c", parent=st["BodyText"], fontSize=7.5, leading=9)
    reviewed = reviewed.copy()
    for c in ("description", "current_status", "risk_age_days", "update_flag", "change_since_last", "overdue_flag"):
        if c not in reviewed: reviewed[c] = ""
    def tbl(df, cols, widths, heads=None, tint=()):
        data = [[Paragraph(f"<b>{h}</b>", ParagraphStyle("h", parent=cell, textColor=colors.white)) for h in (heads or cols)]] + [[Paragraph(esc(str(v if v is not None and v == v else ""))[:420], cell) for v in row] for row in df[cols].values.tolist()]
        t = Table(data, colWidths=widths, repeatRows=1)
        sty = [("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#16324f")), ("GRID", (0, 0), (-1, -1), 0.3, colors.grey), ("VALIGN", (0, 0), (-1, -1), "TOP")]
        for target, source in tint:                                  # colour the target column's cells by the value in the source column
            for i, v in enumerate(df[source].tolist(), 1):
                col = tint_of(v)
                if col: sty.append(("BACKGROUND", (cols.index(target), i), (cols.index(target), i), colors.HexColor(col)))
        t.setStyle(TableStyle(sty)); return t
    def footer(c, d):
        c.saveState(); c.setFont("Helvetica-Oblique", 7); c.setFillColor(colors.grey)
        c.drawString(36, 20, f"{DISCLAIMER}   |   Page {d.page}"); c.restoreState()
    def age_text(x):
        try: return f"{int(float(x['risk_age_days']))} days / {x['current_status']}"
        except Exception: return f"{x['current_status']}"
    doc = SimpleDocTemplate(str(path), pagesize=landscape(letter), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=36)
    s = [Paragraph("Project Risk Analysis Report", st["Title"]), Paragraph(f"<i>{DISCLAIMER}</i>", st["BodyText"]), Spacer(1, 6), Paragraph(esc(summary_line), st["BodyText"]),
         Paragraph("Colour key: <b>Red</b>, <b>Amber</b> and <b>Green</b> show the threat level; grey = closed. Risk age = days from first identified to the latest meeting (or to the closing meeting).", cell), Spacer(1, 8)]
    s += [Paragraph("1. Reviewed risk register (with reasons and source references)", st["Heading2"])]
    if len(reviewed):
        r = reviewed.sort_values("score", ascending=False).copy(); r["ref"] = r.apply(lambda x: f"{x['source']} p.{int(x['page']) if x['page'] == x['page'] and x['page'] is not None else '?'}", axis=1)
        r["age"] = r.apply(age_text, axis=1)
        s += [tbl(r, ["reg_id", "title", "description", "category", "score", "rationale", "rag", "stage", "age", "review_state", "evidence", "ref"], [52, 64, 88, 52, 36, 92, 40, 46, 44, 58, 78, 56],
                  ["Risk ID", "Risk", "Description", "Category", "Score", "Reason for rating", "Threat", "Stage", "Age / Status", "Review", "Evidence (quote)", "Source"], tint=[("score", "rag"), ("rag", "rag")])]
    else: s += [Paragraph("No risks in the register.", st["BodyText"])]
    s += [Spacer(1, 4), Paragraph("<b>Scoring method:</b> Score = probability x impact. Likelihood 1-5 becomes probability 0.1 / 0.3 / 0.5 / 0.7 / 0.9 and is multiplied by impact 1-5 (maximum 4.5). Red: score &gt;= 2.5. Amber: 1.2-2.4. Green: &lt;= 1.0. Closed risks are shown in grey.", cell),
          Paragraph("<b>Impact scale</b> (schedule = delay to substantial completion; cost = % of original contract value; if both apply, the higher rating is used): 1 = up to 2 working days / under 0.25%; 2 = 3-5 days / 0.25-0.5%; 3 = 6-10 days / 0.5-1%; 4 = 11-20 days / 1-2%; 5 = over 20 days or a contract milestone missed / over 2%.", cell)]
    s += [Spacer(1, 8), Paragraph("2. Response plan: risk or issue, response strategy and owner", st["Heading2"])]
    if len(reviewed):
        rp = reviewed.sort_values("score", ascending=False).copy()
        for c in ("type", "response_strategy", "response_action", "owner", "due_date"):
            if c not in rp: rp[c] = ""
        s += [tbl(rp, ["reg_id", "title", "type", "response_strategy", "response_action", "owner", "due_date", "rag"], [70, 120, 40, 70, 250, 90, 56, 36],
                  ["Risk ID", "Risk", "Type", "Strategy", "Response / action (from the minutes)", "Owner", "Due", "Threat"], tint=[("rag", "rag")])]
    s += [Spacer(1, 8), Paragraph("3. Risk updates since the previous meeting, and risks whose response date has passed", st["Heading2"])]
    u = reviewed[(~reviewed["update_flag"].fillna("").isin(["", "No change", "Closed"])) | (reviewed["overdue_flag"].fillna("") != "")] if len(reviewed) else reviewed
    if len(u): s += [tbl(u, ["reg_id", "title", "update_flag", "change_since_last", "overdue_flag", "rag"], [70, 190, 110, 170, 130, 50], ["Risk ID", "Risk", "Update", "What changed", "Response overdue?", "Threat"],
                         tint=[("update_flag", "update_flag"), ("overdue_flag", "overdue_flag"), ("rag", "rag")])]
    else: s += [Paragraph("No changes or overdue responses flagged.", st["BodyText"])]
    s += [Spacer(1, 8), Paragraph("4. Action items (overdue first)", st["Heading2"])]
    if len(actions): s += [tbl(actions, ["project_no", "action_id", "description", "owner", "due_date", "status", "days_overdue", "attention", "source", "page"], [54, 36, 160, 76, 52, 44, 34, 140, 60, 24],
                               ["Project", "ID", "Description", "Owner", "Due", "Status", "Days late", "Needs attention", "Source", "Pg"], tint=[("status", "status"), ("attention", "attention")])]
    s += [Spacer(1, 8), Paragraph("5. Cross-section inconsistencies found in the minutes", st["Heading2"])]
    s += [tbl(inc, ["project_no", "meeting_no", "kind", "detail"], [70, 50, 80, 520], ["Project", "Mtg", "Type", "Detail"])] if len(inc) else [Paragraph("None found.", st["BodyText"])]
    s += [Spacer(1, 8), Paragraph("6. Audit trail: extraction failures, unsupported claims and user corrections", st["Heading2"])]
    if len(audit):
        counts = audit["kind"].value_counts().to_dict(); s += [Paragraph(esc(", ".join(f"{k}: {v}" for k, v in counts.items())), st["BodyText"]), Spacer(1, 4)]
        keep = audit[audit["kind"].isin(["extraction_failure", "unsupported_claim", "user_correction", "low_confidence", "missing_required_field", "validation_failure"])].head(40)
        if len(keep): s += [tbl(keep, ["ts", "kind", "doc_id", "detail"], [90, 90, 160, 390], ["Time", "Type", "Document", "Detail"])]
    doc.build(s, onFirstPage=footer, onLaterPages=footer)

def node_report(state: S):
    out = Path(CFG["OUT_DIR"]); findings, reg, inc, acts = build_frames(state)
    reviewed = order_register(pd.DataFrame(state.get("reviewed_register", [])))
    audit = pd.DataFrame(AUDIT)
    findings.to_csv(out / "risk_findings.csv", index=False); reg.to_csv(out / "risk_register_system.csv", index=False)
    reviewed.to_csv(out / "risk_register_reviewed.csv", index=False); inc.to_csv(out / "inconsistencies.csv", index=False)
    acts.to_csv(out / "action_items.csv", index=False); audit.to_csv(out / "audit_log.csv", index=False)
    n_over = int((acts["status"] == "Overdue").sum()) if len(acts) else 0
    flagged = reviewed[(~reviewed["update_flag"].fillna("").isin(["", "No change", "Closed"])) | (reviewed["overdue_flag"].fillna("") != "")] if "update_flag" in reviewed else reviewed.iloc[0:0]
    flagged.to_csv(out / "risk_updates.csv", index=False)                                   # new/escalated/improved/stale risks and overdue risk responses
    (acts[acts["status"] == "Overdue"] if n_over else acts.iloc[0:0]).to_csv(out / "overdue_actions.csv", index=False)
    line = (f"{len(state['records'])} meeting minute(s) analysed; {len(reviewed)} risks in the reviewed register; {len(acts)} action items tracked ({n_over} overdue); "
            f"{len(flagged)} risk updates or overdue responses flagged; "
            f"{len(inc)} inconsistencies found; review decisions: {summary(state.get('reviewed', []))}.")
    try: make_pdf(out / "risk_report.pdf", reviewed, acts, inc, audit, line)
    except Exception as e: log_event("extraction_failure", None, f"PDF report failed: {e}"); print("PDF report skipped:", e)
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        if len(findings) and findings["meeting_no"].nunique() > 1:
            fig, ax = plt.subplots(figsize=(8, 4))
            for p, g in findings.groupby("project_no"): t = g.groupby("meeting_no")["score"].mean(); ax.plot(t.index, t.values, marker="o", label=p)
            ax.set_xlabel("Meeting #"); ax.set_ylabel("Mean risk score"); ax.set_title("Risk score trend"); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(out / "risk_trends.png", dpi=130); plt.close(fig)
    except Exception: pass
    for rid, rec in state["records"].items():      # cache analysed docs so re-uploads are instant and cross-meeting history persists
        d = next((x for x in state["docs"] if x["doc_id"] == rid), None)
        if d: json.dump(dict(hash=d["hash"], record=rec, assessed=state["assessed"].get(rid, []), actions=state.get("actions", {}).get(rid, [])), open(Path(CFG["CACHE_DIR"]) / f"{rid}.json", "w", encoding="utf-8"), default=str)
    files = sorted(p.name for p in out.iterdir() if p.is_file())
    return {"report": dict(out_dir=str(out), files=files, summary=line), "log": [f"[10 report] exported {len(files)} files to {out}/ | {n_over} overdue action(s), {len(flagged)} risk update(s) flagged"]}

## 11. Build the LangGraph + session API (what a Streamlit app would call)

In [ ]:
def route_after_quality(state: S): return "risk_extraction" if state.get("todo") else "human_review"

def build_graph():
    g = StateGraph(S)
    for name, fn in [("intake", node_intake), ("pdf_extraction", node_pdf_extraction), ("metadata", node_metadata), ("chunk_index", node_chunk_index),
                     ("retrieval", node_retrieval), ("risk_extraction", node_extract_risks), ("classification", node_classify), ("historical_matching", node_match),
                     ("quality_check", node_quality), ("human_review", node_human_review), ("report", node_report)]: g.add_node(name, fn)
    g.add_edge(START, "intake"); g.add_conditional_edges("intake", route_after_intake, {"pdf_extraction": "pdf_extraction", END: END})
    for a, b in [("pdf_extraction", "metadata"), ("metadata", "chunk_index"), ("chunk_index", "retrieval"), ("retrieval", "risk_extraction"), ("risk_extraction", "classification"),
                 ("classification", "historical_matching"), ("historical_matching", "quality_check"), ("human_review", "report")]: g.add_edge(a, b)
    g.add_conditional_edges("quality_check", route_after_quality, {"risk_extraction": "risk_extraction", "human_review": "human_review"}); g.add_edge("report", END)
    return g.compile(checkpointer=MemorySaver())      # swap MemorySaver for SqliteSaver in a Streamlit app to survive restarts
APP = build_graph()

SESSION: Dict[str, Any] = dict(status="none", config=None, props=[], state=None)      # status: none | awaiting_review | final

def list_uploaded(): return sorted(glob.glob(str(Path(CFG["UPLOAD_DIR"]) / "*.pdf")))
def _unique_dest(name, parent=""):
    d = Path(CFG["UPLOAD_DIR"]) / Path(name).name
    return d if not d.exists() else Path(CFG["UPLOAD_DIR"]) / f"{parent}_{Path(name).name}"

def save_uploads(items):
    """items: {filename: bytes} -> saved PDF paths. A .zip is unpacked and every PDF inside it is kept (use from Streamlit's file_uploader)."""
    import io, zipfile
    paths = []
    for name, data in items.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for m in z.namelist():
                    if m.lower().endswith(".pdf") and not Path(m).name.startswith("."):
                        d = _unique_dest(m, Path(m).parent.name); d.write_bytes(z.read(m)); paths.append(str(d))
        elif name.lower().endswith(".pdf"):
            d = Path(CFG["UPLOAD_DIR"]) / Path(name).name; d.write_bytes(data); paths.append(str(d))
    return paths

def add_folder(folder):
    """Add every PDF under a folder (subfolders included), e.g. a mounted Google Drive folder or a local folder."""
    n = 0
    for p in sorted(Path(folder).rglob("*.pdf")):
        d = _unique_dest(p.name, p.parent.name); shutil.copy(p, d); n += 1
    print(f"Added {n} PDF(s) from {folder}. Total uploaded: {len(list_uploaded())}"); return n

def start_analysis(pdf_paths=None, force=False):
    """Run the pipeline up to the human-review pause (or to the end if nothing needs review)."""
    paths = list(pdf_paths or list_uploaded())
    if not paths: raise ValueError("No PDFs uploaded yet.")
    config = {"configurable": {"thread_id": f"run-{int(time.time()*1000)}"}, "recursion_limit": 80}
    APP.invoke({"files": paths, "force": force, "log": []}, config)
    snap = APP.get_state(config); SESSION.update(config=config, state=snap.values)
    if snap.next: SESSION.update(status="awaiting_review", props=snap.tasks[0].interrupts[0].value["proposals"])
    else: SESSION.update(status="final", props=snap.values.get("reviewed", []))
    for line in snap.values.get("log", []): print(line)
    return SESSION

def submit_review(props=None):
    """Resume the paused graph with the project manager's decisions -> final reviewed register + exported report."""
    res = APP.invoke(Command(resume=props if props is not None else SESSION["props"]), SESSION["config"])
    SESSION.update(state=res, status="final", props=res.get("reviewed", []))
    for line in res.get("log", []):
        if line.startswith(("[9b", "[10")): print(line)
    return res

def clear_cache():
    shutil.rmtree(CFG["CACHE_DIR"], ignore_errors=True); Path(CFG["CACHE_DIR"]).mkdir(parents=True, exist_ok=True); REVIEWS.clear(); print("Analysis cache cleared.")

def download_outputs():
    """Zip the outputs folder; downloads it in Colab, shows a link in Jupyter."""
    z = shutil.make_archive(str(Path(CFG["WORK_DIR"]) / "risk_outputs"), "zip", CFG["OUT_DIR"])
    if IN_COLAB:
        from google.colab import files; files.download(z)
    else:
        from IPython.display import FileLink, display; display(FileLink(z))
    return z

def show_graph(app=None, save_path=None):
    """Architecture diagram (solid = normal edge, dashed = conditional edge)."""
    app = app or APP; save_path = save_path or str(Path(CFG["OUT_DIR"]) / "risk_agent_graph.png")
    from IPython.display import Image, display
    try:
        png = app.get_graph().draw_mermaid_png(); open(save_path, "wb").write(png); display(Image(png)); return
    except Exception as e: print(f"Online renderer unavailable ({type(e).__name__}); drawing offline instead.")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    G = app.get_graph()
    pref = ["__start__", "intake", "pdf_extraction", "metadata", "chunk_index", "retrieval", "risk_extraction", "classification", "historical_matching", "quality_check", "human_review", "report", "__end__"]
    names = [n for n in pref if n in G.nodes] + [n for n in G.nodes if n not in pref]; pos = {n: (0, -i * 1.25) for i, n in enumerate(names)}
    fig, ax = plt.subplots(figsize=(6.5, 1.0 + 1.25 * len(names)))
    for n, (x, y) in pos.items():
        end = n.startswith("__")
        ax.add_patch(FancyBboxPatch((x - 1.5, y - 0.3), 3.0, 0.6, boxstyle="round,pad=0.02,rounding_size=" + ("0.3" if end else "0.1"), fc="#bfb6fc" if n == "__end__" else ("#ffffff" if end else "#f2f0ff"), ec="#b8a9f5", lw=1.6, zorder=2))
        ax.text(x, y, n, ha="center", va="center", fontsize=11, zorder=3)
    side = {"R": 0, "L": 0}
    for e in G.edges:
        (x1, y1), (x2, y2) = pos[e.source], pos[e.target]; i, j = names.index(e.source), names.index(e.target); ls = "--" if e.conditional else "-"
        ap = dict(arrowstyle="-|>", lw=1.5, color="#444", ls=ls, mutation_scale=16)
        if j == i + 1: ax.annotate("", xy=(x2, y2 + 0.32), xytext=(x1, y1 - 0.32), arrowprops=ap, zorder=1)
        else:
            sgn = 1 if j > i else -1; key = "R" if sgn == 1 else "L"; side[key] += 1; xo = sgn * (1.5 + 0.5 * side[key])
            ax.plot([sgn * 1.5, xo, xo], [y1, y1, y2], color="#444", lw=1.5, ls=ls, zorder=1); ax.annotate("", xy=(sgn * 1.5, y2), xytext=(xo, y2), arrowprops=ap, zorder=1)
    ax.set_xlim(-3.2, 3.2); ax.set_ylim(-1.25 * len(names) + 0.7, 0.8); ax.axis("off"); fig.tight_layout(); fig.savefig(save_path, dpi=150); plt.show()

## 12. Chat engine: ask risk questions in plain language
`chat("...")` answers from the uploaded minutes (ChromaDB retrieval) **and** the structured risk register, with reasons and source pages.
Say "analyze these minutes" to run the analysis, "export the report" to export. With no API key it falls back to an evidence-only answer.

In [ ]:
CHAT_HISTORY: List[tuple] = []
BROAD = re.compile(r"\b(all|overall|summar|top|list|every|across|trend|status|biggest|main|key|red|amber|open)\b", re.I)
RISKY = re.compile(r"risk|issue|delay|cost|schedule|supplier|supply|safety|quality|permit|action|overdue|owner|due|change order|payment", re.I)

def current_register(project=None):
    """(DataFrame, label). Reviewed register once review is done, otherwise the system's draft."""
    st = SESSION.get("state")
    if not st: return pd.DataFrame(), "none"
    if st.get("reviewed_register") and SESSION["status"] == "final": df, label = pd.DataFrame(st["reviewed_register"]), "reviewed by the project manager"
    else:
        f, *_ = build_frames(st)
        df, label = (f.sort_values("meeting_no").groupby("reg_id").tail(1).copy() if len(f) else pd.DataFrame()), "DRAFT - not yet reviewed"
        if len(df): df["review_state"] = "Draft"
    if project and len(df): df = df[df["project_no"] == project]
    return df, label

def _row_text(i, r):
    pg = r.get("page"); pg = int(pg) if pg == pg and pg is not None else "?"
    return (f"[R{i}] {r['reg_id']}: {r['title']} | category {r['category']} | type {r.get('type')} | response {r.get('response_strategy')}: {r.get('response_action')} | stage {r.get('stage')} | score {r.get('score')} ({r.get('rag')}) | review: {r.get('review_state')} | "
            f"why: {r.get('rationale')} | evidence: \"{str(r.get('evidence'))[:220]}\" | source: {r.get('source')} p.{pg}")

def answer_question(q, project=None):
    if get_vectorstore()._collection.count() == 0: return dict(answer="No minutes are indexed yet. Upload one or more meeting-minute PDFs first, then ask again.", sources=[])
    broad = bool(BROAD.search(q)); k = 10 if broad else max(CFG["TOP_K"], 6)
    hits = search(q, k, **({"project_no": project} if project else {}))
    reg, label = current_register(project); ctx_r, rows = [], []
    act_q, st = bool(re.search(r"action|overdue|owner|due\b|to-?do", q, re.I)), SESSION.get("state")
    acts = []
    if act_q and st:
        acts = [a for a in st.get("action_register", []) if not project or a["project_no"] == project]
        if re.search(r"overdue|late|past due", q, re.I):
            latest = {}
            for r in st["records"].values():
                if r["meeting_date"]: latest[r["project_no"]] = max(latest.get(r["project_no"], ""), r["meeting_date"])
            acts = [a for a in acts if a["status"] == "Overdue" or (a["status"] == "Open" and a["due_date"] and a["due_date"] < latest.get(a["project_no"], ""))]
    if len(reg) and RISKY.search(q) and not (act_q and acts and "risk" not in q.lower()):
        toks = {w for w in norm(q).split() if len(w) > 3}
        scored = reg.assign(_m=reg.apply(lambda r: len(toks & set(norm(f"{r['title']} {r['category']} {r['reg_id']}").split())), axis=1)).sort_values(["_m", "score"], ascending=False)
        rows = scored.head(12 if broad else 5).to_dict("records"); ctx_r = [_row_text(i + 1, r) for i, r in enumerate(rows)]
    ctx_a = [f"[A{i+1}] {a['project_no']} {a['action_id']}: {a['description']} | owner {a['owner']} | due {a['due_date']} | status {a['status']} ({a.get('attention') or 'no flags'}) | across meetings: {a['status_updates']} | source {a['source']} p.{a['page']}" for i, a in enumerate(acts[:15])]
    ctx_s = [f"[S{i+1}] {h['meta']['source']} p.{h['meta']['page']} ({h['meta']['section']}, meeting {h['meta']['meeting_no']}, {h['meta']['project_no']})\n{h['text']}" for i, h in enumerate(hits)]
    sources = [f"{h['meta']['source']} p.{h['meta']['page']}" for h in hits]
    llm_ans = None
    if CFG["_llm"] != "mock":
        system = ("You are a construction project risk assistant. Answer ONLY from the CONTEXT (meeting-minute passages [S#], risk register rows [R#], action items [A#]). "
                  "Be concise and write like a helpful colleague. Structure: 1) Answer, 2) Why (the reasons behind it), 3) Evidence (short quotes with [S#]/[R#]/[A#] tags), 4) Confidence and gaps. "
                  f"If the context does not contain the answer, say so. The risk register is {label}; say so if it is a draft. Never invent numbers. End with: '{DISCLAIMER}'")
        hist = "\n".join(f"User: {a}\nAssistant: {b[:400]}" for a, b in CHAT_HISTORY[-3:])
        llm_ans = llm_text(system, f"Conversation so far:\n{hist}\n\nQuestion: {q}\n\nCONTEXT\nRisk register ({label}):\n" + "\n".join(ctx_r) + "\n\nActions:\n" + "\n".join(ctx_a) + "\n\nPassages:\n" + "\n\n".join(ctx_s))
    if llm_ans:
        used = sorted(set(re.findall(r"\[([SRA]\d+)\]", llm_ans)))
        refs = [f"[{u}] " + (f"{hits[int(u[1:])-1]['meta']['source']} p.{hits[int(u[1:])-1]['meta']['page']}" if u[0] == "S" and int(u[1:]) <= len(hits)
                              else f"{rows[int(u[1:])-1]['source']} p.{rows[int(u[1:])-1]['page']}" if u[0] == "R" and int(u[1:]) <= len(rows)
                              else f"{acts[int(u[1:])-1]['source']} p.{acts[int(u[1:])-1]['page']}" if u[0] == "A" and int(u[1:]) <= len(acts) else "") for u in used]
        refs = [r for r in refs if r.split("] ", 1)[-1].strip()]
        return dict(answer=llm_ans + ("\n\n**Sources:** " + "; ".join(refs) if refs else ""), sources=sources)
    parts = ["**Evidence-only mode** (no LLM key set, so I can show what I found but not write a reasoned answer).\n"]
    if act_q and not ctx_a and st: parts += ["No matching action items found.\n"]
    if ctx_a: parts += ["Action items:"] + ctx_a + [""]
    if ctx_r: parts += [f"Risk register ({label}):"] + ctx_r + [""]
    parts += ["Most relevant passages:"] + [c[:700] for c in ctx_s[:4]] + ["", f"_{DISCLAIMER}_"]
    return dict(answer="\n".join(parts), sources=sources)

def chat(question, project=None):
    """One conversational turn. Returns {'answer': str, 'sources': [...], 'action': None|'review'|'export'}."""
    ql = question.lower()
    if re.search(r"\b(re-?run|force)\b.*\banaly[sz]", ql) or re.search(r"^\s*(please\s+)?(analy[sz]e|run (the )?analysis|go through|assess)\b", ql):
        if not list_uploaded(): return dict(answer="Upload a meeting-minute PDF first, then ask me to analyze it.", sources=[], action=None)
        start_analysis(force=bool(re.search(r"re-?run|force", ql))); n = len([p for p in SESSION["props"] if p["decision"] in ("Pending", "Needs clarification")])
        if SESSION["status"] == "awaiting_review":
            msg = f"I read the minutes and built a risk register. **{n} suggested changes need your review** before I produce the final report. Review them in the panel below, then submit."
            act = "review"
        else: msg, act = f"Analysis complete. The reviewed register and report are in `{CFG['OUT_DIR']}`.", "export"
        return dict(answer=msg, sources=[], action=act)
    if re.search(r"\b(export|download)\b|(generate|create|produce|make|give me)\s+(the |a |my )?(final )?report", ql):
        if SESSION["status"] == "awaiting_review": return dict(answer="The report is produced after your review. Please finish the review panel and submit your decisions first.", sources=[], action="review")
        if SESSION["status"] == "none": return dict(answer="I haven't analyzed any minutes yet. Upload a PDF and say \"analyze these minutes\".", sources=[], action=None)
        return dict(answer=f"The report is ready: `{CFG['OUT_DIR']}/risk_report.pdf` (plus CSVs and the audit log). Call download_outputs() to download everything.", sources=[], action="export")
    r = answer_question(question, project); r["action"] = None; CHAT_HISTORY.append((question, r["answer"])); return r

## 13. Notebook UI helpers (uploader, review panel, chat box). Optional: Streamlit replaces these.

In [ ]:
def pdf_uploader():
    """Colab: opens the file picker. Jupyter: shows an upload widget with a Save button. PDFs are saved to CFG['UPLOAD_DIR']."""
    if IN_COLAB:
        from google.colab import files
        up = files.upload(); paths = save_uploads(up)      # PDFs or a .zip of PDFs
        print(f"Saved {len(paths)} PDF(s). Total uploaded: {len(list_uploaded())}"); return paths
    import ipywidgets as w; from IPython.display import display
    up = w.FileUpload(accept=".pdf,.zip", multiple=True); btn = w.Button(description="Save uploaded PDFs", button_style="success"); out = w.Output()
    def save(_):
        val = up.value; items = {}
        for f in (val.values() if isinstance(val, dict) else val): items[f["name"]] = bytes(f["content"])
        with out: out.clear_output(); print(f"Saved {len(save_uploads(items))} PDF(s). Total uploaded: {len(list_uploaded())}")
    btn.on_click(save); display(w.VBox([up, btn, out]))

def folder_uploader(default_path=""):
    """Add a whole folder of PDFs from a cell. Type/paste the folder path (Colab: mount Drive first, then e.g. /content/drive/MyDrive/my_folder;
    Jupyter: a local path) and click 'Add folder'. Or upload a .zip of the folder. Subfolders are included."""
    import ipywidgets as w; from IPython.display import display
    if IN_COLAB:
        try:
            from google.colab import drive, output; output.enable_custom_widget_manager()
            if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
        except Exception as e: print("Drive not mounted:", e)
    path = w.Text(value=default_path or ("/content/drive/MyDrive/" if IN_COLAB else ""), description="Folder", layout=w.Layout(width="620px"),
                  placeholder="/content/drive/MyDrive/my_minutes" if IN_COLAB else "C:/Users/you/minutes  or  ./minutes")
    add = w.Button(description="Add folder", button_style="success"); out = w.Output()
    def on_add(_):
        with out:
            out.clear_output(); p = Path(path.value.strip().strip('"'))
            if p.is_dir(): add_folder(p)
            else: print("Folder not found:", p, "(in Colab, check the path in the Files panel: right-click the folder > Copy path)")
    add.on_click(on_add); items = [w.HBox([path, add]), out]
    if IN_COLAB:
        zbtn = w.Button(description="...or upload a .zip of the folder", button_style="info")
        def on_zip(_):
            with out:
                out.clear_output(); from google.colab import files
                up = files.upload(); print(f"Saved {len(save_uploads(up))} PDF(s). Total uploaded: {len(list_uploaded())}")
        zbtn.on_click(on_zip); items.insert(1, zbtn)
    else:
        zup = w.FileUpload(accept=".zip", multiple=False, description="Upload .zip"); zb = w.Button(description="Save zip")
        def on_zb(_):
            val = zup.value; its = {f["name"]: bytes(f["content"]) for f in (val.values() if isinstance(val, dict) else val)}
            with out: out.clear_output(); print(f"Saved {len(save_uploads(its))} PDF(s). Total uploaded: {len(list_uploaded())}")
        zb.on_click(on_zb); items.insert(1, w.HBox([zup, zb]))
    display(w.VBox(items))

def review_ui(props, on_submit=None, page_size=8):
    """Review panel: existing vs proposed values, reason, source page; edit / approve / reject; unresolved items stay flagged."""
    import ipywidgets as w; from IPython.display import display
    widgets = {}
    def row(p):
        if p["pid"] in widgets: return widgets[p["pid"]]
        ex, pr = p["existing"], p["proposed"]; hl = lambda k: " style='background:#fff3cd'" if k in p["changed"] else ""
        cells = "".join(f"<tr><td>{k}</td><td>{'-' if ex is None else ex[k]}</td><td{hl(k)}>{pr[k]}</td></tr>" for k in REVIEW_FIELDS)
        warn = (f"<div style='color:#b00020'>&#9888; Needs clarification: {p['issues']}</div>" if p["issues"] else "") + "".join(f"<div style='color:#b00020'>&#9888; {n}</div>" for n in p["doc_notes"])
        pg = p.get("page"); pg = int(pg) if pg is not None else "?"
        head = w.HTML(f"<b>{p['reg_id']}</b> &middot; meeting {p['meeting_no']} &middot; <i>{p['kind']}</i><br><b>{p['title']}</b> ({p['category']}){warn}"
                      f"<table style='margin-top:4px'><tr><th>Field</th><th>Existing</th><th>Proposed</th></tr>{cells}</table>"
                      f"<small><b>Reason:</b> {p['rationale']}<br><b>Evidence:</b> &ldquo;{p['evidence']}&rdquo; <i>({p['source']}, p.{pg})</i></small>")
        stage = w.Dropdown(options=STAGES, value=pr["stage"] if pr["stage"] in STAGES else "open", description="stage", layout=w.Layout(width="200px"))
        L = w.IntSlider(value=int(pr["likelihood"]), min=1, max=5, description="likelihood", layout=w.Layout(width="260px"))
        I = w.IntSlider(value=int(pr["impact"]), min=1, max=5, description="impact", layout=w.Layout(width="260px"))
        owner = w.Text(value=pr["owner"] or "", description="owner", layout=w.Layout(width="260px"))
        typ = w.Dropdown(options=TYPES, value=pr.get("type") if pr.get("type") in TYPES else "Risk", description="type", layout=w.Layout(width="200px"))
        strat = w.Dropdown(options=STRATEGIES, value=pr.get("response_strategy") if pr.get("response_strategy") in STRATEGIES else "Not stated", description="response", layout=w.Layout(width="260px"))
        dec = w.ToggleButtons(options=DECISIONS, value=p["decision"], style={"button_width": "120px"}); com = w.Text(value=p["comment"], placeholder="comment (optional)", layout=w.Layout(width="420px"))
        def sync(*_):
            p["proposed"].update(stage=stage.value, likelihood=L.value, impact=I.value, owner=owner.value or None, type=typ.value, response_strategy=strat.value); p["decision"], p["comment"] = dec.value, com.value; refresh_summary()
        for x in (stage, L, I, owner, typ, strat, dec, com): x.observe(sync, "value")
        widgets[p["pid"]] = w.VBox([head, w.HBox([stage, L, I, owner]), w.HBox([typ, strat]), dec, com], layout=w.Layout(border="1px solid #ccc", padding="6px", margin="4px 0")); return widgets[p["pid"]]
    show = w.Dropdown(options=["Unresolved only", "All", "Needs clarification"], description="Show"); proj = w.Dropdown(options=["All"] + sorted({str(p["project_no"]) for p in props}), description="Project")
    page = w.BoundedIntText(value=1, min=1, max=1, description="Page"); summ, body = w.HTML(), w.VBox()
    approve_all = w.Button(description="Approve all shown without issues", button_style="info"); submit = w.Button(description="Submit decisions & finish", button_style="success"); msg = w.Output()
    def filtered():
        ps = [p for p in props if proj.value in ("All", str(p["project_no"]))]
        if show.value == "Unresolved only": ps = [p for p in ps if p["decision"] in ("Pending", "Needs clarification")]
        if show.value == "Needs clarification": ps = [p for p in ps if p["decision"] == "Needs clarification"]
        return ps
    def refresh_summary(*_): summ.value = " | ".join(f"<b>{k}</b>: {v}" for k, v in summary(props).items())
    def render(*_):
        ps = filtered(); page.max = max(1, -(-len(ps) // page_size)); page.value = min(page.value, page.max)
        body.children = [row(p) for p in ps[(page.value - 1) * page_size: page.value * page_size]]; refresh_summary()
    def do_approve_all(_):
        for p in filtered()[(page.value - 1) * page_size: page.value * page_size]:
            if p["decision"] == "Pending" and not p["issues"]: widgets.pop(p["pid"], None); p["decision"] = "Approved"
        render()
    def do_submit(_):
        with msg:
            msg.clear_output(); print("Submitting decisions and resuming the graph ...")
            if on_submit: on_submit(props); print("Done. The final report has been exported.")
    for x in (show, proj, page): x.observe(render, "value")
    approve_all.on_click(do_approve_all); submit.on_click(do_submit)
    display(w.VBox([w.HTML("<i>" + DISCLAIMER + "</i>"), w.HBox([show, proj, page]), summ, body, w.HBox([approve_all, submit]), msg])); render()

def chat_ui(project=None):
    """Chat box: ask questions, say 'analyze these minutes', then review and say 'export the report'."""
    import ipywidgets as w; from IPython.display import display, Markdown, clear_output
    log = w.Output(layout=w.Layout(border="1px solid #ccc", padding="6px", max_height="460px", overflow="auto")); review_area = w.Output()
    box = w.Text(placeholder="e.g. 'Analyze these minutes'  |  'What are the main cost risks and why?'  |  'Which actions are overdue?'", layout=w.Layout(width="82%"))
    send = w.Button(description="Send", button_style="primary")
    def done(props):
        submit_review(props)
        with log: display(Markdown(f"**Assistant:** Review saved. Final register and report exported to `{CFG['OUT_DIR']}` (risk_report.pdf, risk_register_reviewed.csv, audit_log.csv). Run `download_outputs()` to download them."))
    def on_send(_):
        q = box.value.strip()
        if not q: return
        box.value = ""
        with log:
            display(Markdown(f"**You:** {q}")); r = chat(q, project); display(Markdown(f"**Assistant:** {r['answer']}"))
        if r.get("action") == "review" and SESSION["status"] == "awaiting_review":
            with review_area: clear_output(); review_ui(SESSION["props"], on_submit=done)
    send.on_click(on_send)
    try: box.on_submit(on_send)
    except Exception: pass
    display(w.VBox([w.HTML("<b>Risk assistant</b> <i>(" + DISCLAIMER + ")</i>"), log, w.HBox([box, send]), review_area]))

---
# NOTEBOOK DEMO: run these cells in order (they are skipped automatically when this file is used as a `.py` engine)

### Step A. (Optional) add your API key
For real answers add `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`). **Colab:** click the key icon (Secrets), add it, enable notebook access, then re-run this cell.
**Jupyter:** uncomment the `set_api_key` line. Without a key the system still runs in evidence-only mode (rule-based extraction, passages instead of written answers).

In [ ]:
set_api_key("OPENAI_API_KEY", "put your API key here")        # or: set_api_key("ANTHROPIC_API_KEY", "sk-ant-...")
resolve_providers()

### Step B. Add your meeting-minute PDF(s)
**Option 1:** run the next cell and select several PDFs (Ctrl/Cmd+A in the picker selects every file in a folder) **or upload a .zip of your folder**.
**Option 2 (best for hundreds of files):** put the folder in Google Drive, then use Step B2.
Upload several meetings of the same project to see cross-meeting status updates.

In [ ]:
if IN_NOTEBOOK: UPLOADED = pdf_uploader()

### Step B2. Add a whole folder
Run the next cell, paste the folder path (Colab: a Drive folder such as `/content/drive/MyDrive/my_minutes`; Jupyter: a local path) and click **Add folder**. Or upload a .zip of the folder. Subfolders are included.

In [ ]:
if IN_NOTEBOOK: folder_uploader()

### Step C. Chat with the system
Try: **"Analyze these minutes"**, then review the suggested changes in the panel and submit. Afterwards ask things like
*"What are the main cost risks and why?"*, *"Which action items are overdue?"*, *"How did the cement supply risk change across meetings?"*, *"Export the report"*.

In [ ]:
if IN_NOTEBOOK: chat_ui()

### Alternative to the chat box: run the steps yourself
(Skip this if you used the chat box. Re-running asks for review again only for items not yet decided.)

In [ ]:
#if IN_NOTEBOOK and False:       # change False to True to use
    #start_analysis()            # runs until the human-review pause
    #if SESSION["status"] == "awaiting_review": review_ui(SESSION["props"], on_submit=submit_review)

### Step D. Results and downloads

In [ ]:
if IN_NOTEBOOK and SESSION["status"] == "final":
    display(pd.DataFrame(SESSION["state"]["reviewed_register"]).head(20))       # reviewed risk register with reasons and sources
    print(os.listdir(CFG["OUT_DIR"]))
    # download_outputs()   # zips and downloads risk_report.pdf, CSVs, audit_log.csv ...

---
## Excel export cell (the one that defines `export_excel`)

In [ ]:

def export_excel(out_dir=None, filename="risk_report.xlsx"):
    """Combine the CSV outputs into ONE formatted Excel workbook: frozen header, filters, wrapped text, and Red/Amber/Green colour coding."""
    from pathlib import Path
    from datetime import datetime
    import pandas as pd
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    out = Path(out_dir or CFG["OUT_DIR"])
    sheets = [("Risk register", "risk_register_reviewed.csv"), ("Risk updates", "risk_updates.csv"), ("Action items", "action_items.csv"), ("Overdue actions", "overdue_actions.csv"),
              ("All findings", "risk_findings.csv"), ("Inconsistencies", "inconsistencies.csv"), ("Review decisions", "review_decisions.csv"), ("Audit log", "audit_log.csv")]
    pretty = dict(reg_id="Risk ID", project_no="Project", rag="Threat level", risk_age_days="Risk age (days)", first_identified="First identified", current_status="Current status",
                  update_flag="Update since last meeting", change_since_last="What changed", overdue_flag="Response overdue?", due_date="Due date", review_state="Review state",
                  unresolved_items="Unresolved items", days_overdue="Days overdue", attention="Needs attention", action_id="Action ID", status_in_minutes="Status in minutes", type="Type (Risk / Issue)", response_strategy="Response strategy", response_action="Response / action")
    tint_by = {"score": "rag", "rag": "rag", "status": "status", "current_status": "current_status", "update_flag": "update_flag", "overdue_flag": "overdue_flag", "attention": "attention"}   # column -> column whose value sets the colour
    path = out / filename
    try: open(path, "ab").close()
    except PermissionError: path = out / f"{Path(filename).stem}_{datetime.now():%H%M%S}.xlsx"; print("The old workbook is open, so saving a new copy instead.")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for name, f in sheets:
            p = out / f
            if not p.exists(): continue
            try: df = pd.read_csv(p)
            except Exception: continue
            if df.empty: continue
            df.to_excel(xw, sheet_name=name, index=False); ws = xw.sheets[name]
            for i, col in enumerate(df.columns, 1):
                h = ws.cell(row=1, column=i); h.value = pretty.get(col, col.replace("_", " ").capitalize())
                h.font = Font(bold=True, color="FFFFFF"); h.fill = PatternFill("solid", fgColor="16324F"); h.alignment = Alignment(wrap_text=True, vertical="center")
                q = df[col].map(lambda v: "" if pd.isna(v) else str(v)).str.len().quantile(0.9)      # empty columns are fine
                width = min(60, max(10, int(q) + 2, len(h.value) + 2))
                ws.column_dimensions[get_column_letter(i)].width = width
                for cell in ws[get_column_letter(i)][1:]: cell.alignment = Alignment(wrap_text=True, vertical="top")
                if col == "score":
                    for cell in ws[get_column_letter(i)][1:]: cell.number_format = "0.0"
                if col in tint_by and tint_by[col] in df.columns:
                    for r, v in enumerate(df[tint_by[col]].tolist(), 2):
                        c = tint_of(v)
                        if c: ws.cell(row=r, column=i).fill = PatternFill("solid", fgColor=c.lstrip("#").upper())
            ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
        key = pd.DataFrame([["Formula", "Score = probability x impact"], ["Probability", "likelihood 1 / 2 / 3 / 4 / 5 = 0.1 / 0.3 / 0.5 / 0.7 / 0.9"], ["Impact", "1 (very low) to 5 (very high)"], ["Range", "0.1 to 4.5"],
                            ["Red", "score >= 2.5: act now; report to the project manager this week"], ["Amber", "score 1.2 to 2.4: assign an owner and a response; watch it at every meeting"],
                            ["Green", "score <= 1.0: monitor"], ["Closed", "shown instead of a colour once the minutes record the risk as resolved"],
                            ["Impact 1 (very low)", "schedule <= 2 working days  |  cost < 0.25% of original contract value"], ["Impact 2 (low)", "3-5 working days  |  0.25-0.5%"],
                            ["Impact 3 (moderate)", "6-10 working days  |  0.5-1%"], ["Impact 4 (high)", "11-20 working days  |  1-2%"],
                            ["Impact 5 (very high)", "> 20 working days or a contract milestone missed  |  > 2%"], ["Impact rule", "if a risk affects both schedule and cost, the higher rating is used"],
                            ["Type", "Issue = already happening or certain; Risk = may still happen. Once a risk has become an issue it stays an issue in the register"],
                            ["Response strategy", "Avoid, Transfer, Mitigate, Accept or Escalate, taken from the action items in the minutes; 'Not stated' if none"]], columns=["Item", "Rule"])
        key.to_excel(xw, sheet_name="Scoring key", index=False); ks = xw.sheets["Scoring key"]
        ks.column_dimensions["A"].width = 22; ks.column_dimensions["B"].width = 100
        for c in ks[1]: c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="16324F")
        for r, name in ((6, "Red"), (7, "Amber"), (8, "Green"), (9, "Closed")): ks.cell(row=r, column=1).fill = PatternFill("solid", fgColor=tint_of(name).lstrip("#").upper())
    print("Saved", path)
    return path

In [ ]:
xlsx_path = export_excel()
# Colab: uncomment to download it
# from google.colab import files; files.download(str(xlsx_path))
#```
#Then call `export_excel()` as before.

### Architecture diagram

In [ ]:
if IN_NOTEBOOK: show_graph()

## Using this as the Streamlit engine
Export the engine (notebook demo cells are skipped automatically because `IN_NOTEBOOK` is False in a script):
```
jupyter nbconvert --to script risk_analyzer.ipynb        # creates risk_analyzer.py  (a ready-made risk_engine.py is also provided)
```
Then in Streamlit: `import risk_engine as eng`, save uploads with `eng.save_uploads({name: bytes})`, call `eng.start_analysis()`,
show `eng.SESSION["props"]` in a review screen, call `eng.submit_review(props)`, answer questions with `eng.chat(question)`,
and offer the files in `eng.CFG["OUT_DIR"]` for download. Set `eng.INSTALL_DEPS = False` in production and swap `MemorySaver` for a SQLite checkpointer.